# Aksara Nusantara: AutoML versi 2, perbandingan backbone yang lengkap

INFEST XII 2026. Tujuh kelas aksara (bali, jawa, jawi, lampung, lontara, pegon,
sunda), penilaian macro-F1, 3.771 citra latih dan 1.220 citra uji.

**Apa yang dijawab notebook ini.** AutoML versi 1 menyapu 19 backbone sebagai
probe beku dan memberi satu tabel peringkat. Tabel itu berguna, tetapi ia
meninggalkan beberapa lubang yang baru terlihat setelah hasilnya dibaca.
Hanya lima kandidat teratas yang ditala, padahal peringkat mentah terbukti
menyesatkan (dinov3-b16 tampak juara sebelum ditala, lalu turun ke urutan
tiga). Tidak ada selang kepercayaan, jadi selisih 0,003 antara dua baris tidak
bisa dibedakan dari keberuntungan. Setiap kandidat hanya diwakili satu angka,
sehingga kita tidak tahu kelas mana yang ia kuasai, populasi citra mana yang
ia lemah, berapa biayanya, atau apakah kesalahannya berbeda dari model lain.

Versi 2 menutup lubang-lubang itu. Perbedaannya dengan versi 1 dirangkum di
tabel berikut.

| Aspek | AutoML v1 | AutoML v2 |
|---|---|---|
| Jumlah backbone | 19 | 333: 43 pilihan tangan ditambah 290 dari katalog timm, 150 lebih keluarga |
| Penalaan probe | 5 teratas saja, 6 setelan | semua kandidat 12 setelan, lalu 40 teratas 30 setelan (2 pooling x 3 normalisasi x 5 nilai C) |
| Bias penalaan | tidak ditangani | setelan dipilih di satu pembagian fold, lalu diukur ulang di dua pembagian lain |
| Jenis kepala | regresi logistik | regresi logistik, kNN kosinus, pusat kelas, MLP kecil |
| Metrik per kandidat | popf1 | 20 lebih kolom: F1 per kelas, blok vs strip, akurasi, log loss, kalibrasi, pertukaran jawi-pegon, dan lainnya |
| Ketidakpastian | tidak ada | selang bootstrap 95%, simpangan antar pembagian fold, uji berpasangan terhadap juara |
| Biaya | detik ekstraksi | parameter, citra per detik, VRAM puncak |
| Keberagaman | fusi greedy | matriks tumpang tindih kesalahan, ketidaksepakatan, dan untung campur untuk setiap pasangan, plus ringkasan per keluarga |
| Lintas sesi | tidak ada | fitur dan hasil penalaan disimpan per backbone, sesi berikutnya melanjutkan |
| Jurang domain | tidak ada | AUC train lawan test di ruang fitur tiap backbone |
| Konfirmasi fine-tune | 2 kandidat | hingga 4 kandidat dari keluarga berbeda, plus korelasi peringkat |
| Submission | tidak ada | `submission.csv` dari ensemble probe terbaik |

**Hal yang perlu dipegang sejak awal.** Angka di notebook ini adalah skor probe
beku, bukan skor fine-tune dan bukan skor papan. Di data ini SigLIP2 @224
mencetak 0,9250 sebagai probe, 0,9735 setelah fine-tune penuh, dan 0,87749 di
papan. Yang dipercaya dari probe adalah urutannya, dan urutan itu pernah
terbukti bertahan sampai fine-tune (v11 ke v12, lalu sonde DINOv3 di v14).
Karena itu `submission.csv` di akhir notebook hampir pasti lebih rendah dari
berkas v15 (0,90090) dan tidak dimaksudkan untuk menggantikannya.

**Waktu jalan.** Perkiraan di Kaggle T4 untuk ke-333 backbone adalah 7 sampai 9
jam dalam satu sesi: sekitar 3 sampai 4 jam ekstraksi fitur, 1,5 jam penalaan
dan metrik, dan sekitar 1 jam konfirmasi fine-tune, sisanya penyiapan dan
gambar. Itu di bawah batas sesi Kaggle 12 jam, dan anggaran bawaannya 11 jam.
Kalau sesi tetap terpotong, tidak ada yang hilang: setiap backbone yang selesai
langsung disimpan, dan Bagian 1.3 menjelaskan cara melanjutkan di sesi
berikutnya. Untuk menjalankan hanya 43 kandidat pilihan tangan (sekitar 4
jam), set `INFEST_KANDIDAT=kurasi`.

Disarankan menjalankan notebook ini di Kaggle lewat **Save Version, Save & Run
All (Commit)**, bukan sesi interaktif. Dengan cara itu notebook berjalan di
latar belakang sampai selesai tanpa perlu peramban tetap terbuka, dan seluruh
isi `/kaggle/working` tersimpan sebagai output versi tersebut.

---
# Bagian 1. Penyiapan

### 1.1 Pustaka

Versi timm harus 1.0.20 ke atas. Tag `v2_webli` milik SigLIP2 baru ada sejak
Februari 2025, arsitektur `dinov3` sejak September 2025, dan Perception
Encoder (`vit_pe_*`) serta AIMv2 juga termasuk tambahan yang relatif baru.
timm bawaan Kaggle maupun Colab bisa tertinggal, dan gejalanya adalah pesan
`Invalid pretrained tag` atau `Unknown model` saat kandidat dimuat.

In [ ]:
# =============================================================================
# PASANG timm YANG CUKUP BARU
# =============================================================================
# Sel ini juga membuang timm lama dari memori Python. Sekali sebuah modul sudah
# diimpor, pip install saja tidak menggantikannya di sesi yang sedang berjalan,
# dan kita tidak ingin harus me-restart runtime hanya karena itu.
#
# Di Kaggle, setelan Internet harus ON (panel kanan, Session options). Tanpa
# itu pip tidak bisa memasang apa pun dan bobot pra-latih tidak bisa diunduh.
import subprocess, sys, importlib, os

if os.environ.get("INFEST_SKIP_PIP", "0") != "1":
    print("memasang timm terbaru...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                        "timm>=1.0.20"], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:]); print(r.stderr[-2000:])
        raise SystemExit("pemasangan timm gagal, lihat pesan di atas")
    for _m in [k for k in list(sys.modules) if k == "timm" or k.startswith("timm.")]:
        del sys.modules[_m]
    importlib.invalidate_caches()
import timm
print(f"timm: {timm.__version__}")

In [ ]:
import os, sys, glob, math, time, json, random, zipfile, hashlib, inspect, warnings, io
from collections import OrderedDict
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from PIL import Image, ImageFile, ImageFilter, ImageEnhance
ImageFile.LOAD_TRUNCATED_IMAGES = True
import matplotlib
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as T
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix
from scipy.stats import spearmanr, kendalltau, binomtest
try:
    from IPython.display import display
except Exception:
    display = print
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 80)

### 1.2 Konfigurasi

Semua keputusan yang bisa diubah dikumpulkan di satu sel. Bagian terbesarnya
adalah daftar `CANDIDATES`. Setiap kandidat punya `fam` (keluarga arsitektur
atau cara pra-latih, dipakai untuk menilai keberagaman), `prio` (urutan kerja,
yang kecil dikerjakan lebih dulu supaya kalau waktu habis yang hilang adalah
kandidat paling spekulatif), dan `mult` (perkiraan kasar biaya relatif
terhadap ViT-B pada resolusi yang sama, dipakai gubernur waktu sebelum
kecepatan sungguhannya terukur).

Daftar ini memuat ke-19 kandidat versi 1 tanpa perubahan, supaya kedua tabel
bisa dibandingkan baris per baris, ditambah 24 kandidat baru. Kandidat baru
dipilih untuk menjawab pertanyaan yang tersisa dari versi 1, bukan karena
populer. Pertanyaan-pertanyaan itu dijelaskan di komentar sel.

In [ ]:
SMOKE = os.environ.get("INFEST_SMOKE", "0") == "1"
SEED  = 42

BUDGET = dict(
    total_h   = float(os.environ.get("INFEST_BUDGET_H", "11.0")),
    reserve_h = 0.20,   # sisa yang tidak boleh disentuh, untuk tabel dan submission
    probe_h   = 2.00,   # dicadangkan untuk penalaan dan metrik setelah ekstraksi
    safety    = 1.25,
)

# -----------------------------------------------------------------------------
# KANDIDAT
# -----------------------------------------------------------------------------
# Pertanyaan yang dijawab kandidat baru:
#
#  (a) Apakah UKURAN membantu di keluarga yang sudah menang? SigLIP2 so400m dan
#      DINOv3 ViT-S serta ViT-L mengisi sumbu ukuran yang di v1 cuma diwakili
#      siglip2-L. Kalau so400m jauh di atas base, itu alasan untuk mencari GPU
#      yang lebih besar, bukan alasan untuk menambah trik.
#  (b) Apakah kurva RESOLUSI yang naik di SigLIP2 (224 < 256 < 384 < 512) juga
#      berlaku di DINOv3? v1 hanya mengukur DINOv3 di 384. Kalau DINOv3 juga naik
#      di 512, anggota DINOv3 di v15 punya ruang yang belum dipakai.
#  (c) Apakah cara POOLING penting? SigLIP2 memakai attention pooling (MAP),
#      varian `gap` memakai rata-rata token. Selain itu setiap kandidat diukur
#      dengan dua pooling sekaligus dari satu lintasan maju (lihat Bagian 3).
#  (d) Keluarga yang belum pernah diuji sama sekali: Perception Encoder (Meta
#      2025), AIMv2 (Apple, pra-latih autoregresif), CLIP OpenAI dan LAION,
#      EVA02-CLIP, SigLIP generasi pertama, CAFormer, MaxViT, EfficientNetV2,
#      DeiT3, RegNetY-SWAG, dan EfficientViT. Semakin banyak keluarga yang
#      terukur, semakin besar peluang menemukan pasangan ensemble yang
#      kesalahannya benar-benar berbeda dari SigLIP2.
#  (e) Satu kandidat sengaja dari domain yang jauh: DINOv3 yang dilatih di citra
#      satelit. Ia pembanding untuk pertanyaan "apakah data pra-latih penting,
#      atau arsitekturnya saja".
def K(tag, name, h, fam, prio, mult=1.0, note="", src="timm", w=None):
    return dict(tag=tag, name=name, h=h, w=w or h, fam=fam, prio=prio, mult=mult,
                note=note, src=src)

CANDIDATES = [
    # ---- jangkar: angka lamanya diketahui, jadi dikerjakan paling dulu -----
    K("siglip2-224", "vit_base_patch16_siglip_224.v2_webli", 224, "siglip2", 0,
      note="jangkar v1: juara lama, fine-tune 0,9735, LB 0,87749"),
    K("siglip2-384", "vit_base_patch16_siglip_384.v2_webli", 384, "siglip2", 0,
      note="jangkar v1: anggota 0 v13 sampai v15"),
    K("dinov3-b16", "vit_base_patch16_dinov3.lvd1689m", 384, "dinov3", 0,
      note="jangkar v1: anggota 1 v14 dan v15"),
    K("cnx-v7-384", "convnext_tiny.in12k_ft_in1k", 384, "convnext-sup", 0, 0.6,
      note="jangkar v1: backbone v6 dan v7, LB 0,839"),
    # ---- kandidat v1 lainnya, dibawa apa adanya --------------------------
    K("siglip2-512", "vit_base_patch16_siglip_512.v2_webli", 512, "siglip2", 1,
      note="v1: popf1_tuned 0,9472, tertinggi"),
    K("siglip2-256", "vit_base_patch16_siglip_256.v2_webli", 256, "siglip2", 1),
    K("siglip2L-256", "vit_large_patch16_siglip_256.v2_webli", 256, "siglip2", 2, 3.2),
    K("siglip2L-384", "vit_large_patch16_siglip_384.v2_webli", 384, "siglip2", 2, 3.2),
    K("dinov3-cnx-b", "convnext_base.dinov3_lvd1689m", 384, "dinov3-cnx", 1, 1.1),
    K("eva02-b448", "eva02_base_patch14_448.mim_in22k_ft_in22k_in1k", 448, "eva02", 2, 1.2),
    K("cnxv2-b384", "convnextv2_base.fcmae_ft_in22k_in1k_384", 384, "convnext-sup", 3, 1.6),
    K("dinov2-b14", "vit_base_patch14_reg4_dinov2.lvd142m", 224, "dinov2", 2, 1.3,
      note="patch 14: resolusi harus habis dibagi 14"),
    K("beitv2-b224", "beitv2_base_patch16_224.in1k_ft_in22k", 224, "beit", 3),
    K("swinv2-b384", "swinv2_base_window12to24_192to384.ms_in22k_ft_in1k", 384, "swin", 3, 2.5),
    K("cnx-clip-384", "convnext_base.clip_laion2b_augreg_ft_in12k_in1k_384", 384, "clip", 2, 1.1),
    K("hiera-b224", "hiera_base_224.mae_in1k_ft_in1k", 224, "hiera", 4),
    K("mae-mss", "davanstrien/vit-manuscripts", 224, "manuskrip-hf", 4, src="hf",
      note="v1: 0,3722, jauh di bawah v11 (0,8904); diukur ulang untuk memastikan"),
    K("vit-iiif", "davanstrien/iiif_manuscript_vit", 224, "manuskrip-hf", 4, src="hf"),
    K("arab-mss", "Ik45/arabic-manuscript-classifier", 224, "manuskrip-hf", 3, src="hf",
      note="manuskrip Arab: relevan untuk pasangan jawi dan pegon"),
    # ---- (a) ukuran ------------------------------------------------------
    K("dinov3-s16", "vit_small_patch16_dinov3.lvd1689m", 384, "dinov3", 1, 0.4,
      note="baru: DINOv3 kecil, 21 juta parameter"),
    K("dinov3-L16", "vit_large_patch16_dinov3.lvd1689m", 384, "dinov3", 2, 3.2,
      note="baru: DINOv3 besar, 300 juta parameter"),
    K("siglip2-so400m-384", "vit_so400m_patch16_siglip_384.v2_webli", 384, "siglip2", 2, 5.0,
      note="baru: SigLIP2 so400m, versi terbesar yang masuk akal di T4"),
    K("siglip2-so400m-512", "vit_so400m_patch16_siglip_512.v2_webli", 512, "siglip2", 5, 5.0,
      note="baru: so400m di resolusi tertinggi, paling mahal di daftar"),
    # ---- (b) resolusi di luar SigLIP2 ------------------------------------
    K("dinov3-b16-256", "vit_base_patch16_dinov3.lvd1689m", 256, "dinov3", 2,
      note="baru: kurva resolusi DINOv3"),
    K("dinov3-b16-512", "vit_base_patch16_dinov3.lvd1689m", 512, "dinov3", 1,
      note="baru: kurva resolusi DINOv3, pertanyaan langsung untuk v15"),
    K("dinov3-cnx-s", "convnext_small.dinov3_lvd1689m", 384, "dinov3-cnx", 3, 0.7),
    K("dinov3-cnx-b-512", "convnext_base.dinov3_lvd1689m", 512, "dinov3-cnx", 3, 1.1),
    # ---- (c) pooling ------------------------------------------------------
    K("siglip2-gap-384", "vit_base_patch16_siglip_gap_384.v2_webli", 384, "siglip2", 2,
      note="baru: SigLIP2 dengan rata-rata token, bukan attention pooling"),
    K("siglip1-384", "vit_base_patch16_siglip_384.webli", 384, "siglip1", 2,
      note="baru: SigLIP generasi pertama, memisahkan efek v2 dari efek keluarga"),
    # ---- (d) keluarga yang belum pernah diuji ------------------------------
    K("pe-core-b384", "vit_pe_core_base_patch16_224.fb", 384, "pe", 1,
      note="baru: Perception Encoder core (Meta 2025), dinaikkan ke 384"),
    K("pe-spatial-b512", "vit_pe_spatial_base_patch16_512.fb", 512, "pe", 2,
      note="baru: PE spatial, dilatih untuk fitur rapat, cocok untuk goresan"),
    K("pe-core-L336", "vit_pe_core_large_patch14_336.fb", 336, "pe", 4, 3.2),
    K("aimv2-L224", "aimv2_large_patch14_224.apple_pt_dist", 224, "aimv2", 3, 3.2,
      note="baru: AIMv2 (Apple), pra-latih autoregresif citra dan teks"),
    K("aimv2-L448", "aimv2_large_patch14_448.apple_pt", 448, "aimv2", 5, 3.2),
    K("clip-oai-384", "vit_base_patch16_clip_384.openai_ft_in12k_in1k", 384, "clip", 2,
      note="baru: CLIP OpenAI lalu IN12k"),
    K("clip-laion-384", "vit_base_patch16_clip_384.laion2b_ft_in12k_in1k", 384, "clip", 3),
    K("eva02-clip-b224", "eva02_base_patch16_clip_224.merged2b", 224, "eva02", 3),
    K("caformer-b384", "caformer_b36.sail_in22k_ft_in1k_384", 384, "metaformer", 3, 1.8),
    K("maxvit-b384", "maxvit_base_tf_384.in21k_ft_in1k", 384, "maxvit", 4, 3.0),
    K("effv2-m384", "tf_efficientnetv2_m.in21k_ft_in1k", 384, "efficientnet", 3, 1.0),
    K("deit3-b384", "deit3_base_patch16_384.fb_in22k_ft_in1k", 384, "deit", 3),
    K("regnety160-384", "regnety_160.swag_ft_in1k", 384, "regnet", 4, 1.5,
      note="baru: RegNetY dilatih lemah-terawasi di 3,6 miliar citra Instagram"),
    K("effvit-l3-384", "efficientvit_l3.r384_in1k", 384, "efficientvit", 4, 1.0),
    # ---- (e) domain pra-latih yang jauh -----------------------------------
    K("dinov3-L16-sat", "vit_large_patch16_dinov3.sat493m", 384, "dinov3-sat", 5, 3.2,
      note="baru: DINOv3 citra satelit, pembanding domain pra-latih"),
]

# -----------------------------------------------------------------------------
# KATALOG LUAS: sekitar 290 backbone tambahan dari timm
# -----------------------------------------------------------------------------
# Daftar di atas disusun tangan untuk menjawab pertanyaan tertentu. Katalog di
# bawah melengkapinya menjadi sekitar 330 backbone, dipilih dengan aturan yang
# tertulis, bukan dengan selera:
#   1. semua bobot pra-latih di timm 1.0.30 (1.807 bobot, 961 arsitektur),
#   2. dibuang: model bahasa dan multimodal, SAM, model uji, ukuran di atas 650
#      juta parameter, dan beberapa keluarga lama yang terbukti jauh tertinggal
#      untuk klasifikasi citra (VGG, MLP-Mixer, ResMLP, gMLP, HaloNet),
#   3. untuk setiap arsitektur diambil SATU bobot, dengan prioritas pra-latih
#      terkaya: DINOv3, SigLIP2, SigLIP, PE, AIMv2, CLIP, lalu ImageNet-22k,
#      ImageNet-12k, dan terakhir ImageNet-1k,
#   4. resolusi mengikuti resolusi asli bobot itu, dibatasi 224 sampai 512,
#   5. dari setiap keluarga diambil paling banyak dua ukuran (satu di sekitar 60
#      juta parameter dan satu ukuran lain) supaya ke-139 keluarga terwakili
#      dan tidak ada keluarga yang mendominasi daftar.
# Setiap nama sudah dicoba dibuat dan dijalankan maju sekali di CPU dengan bobot
# acak sebelum dimasukkan ke sini. Kolomnya: tag, nama timm, resolusi,
# keluarga, juta parameter.
KATALOG = [
    ('aimv2_large_patch14_336', 'aimv2_large_patch14_336.apple_pt_dist', 336, 'aimv', 309.5),
    ('bat_resnext26ts', 'bat_resnext26ts.ch_in1k', 256, 'bat', 8.7),
    ('beit_base_patch16_224', 'beit_base_patch16_224.in22k_ft_in22k', 224, 'beit', 85.8),
    ('beit3_base_patch16_224', 'beit3_base_patch16_224.in22k_ft_in1k', 224, 'beit', 85.9),
    ('beitv2_large_patch16_224', 'beitv2_large_patch16_224.in1k_ft_in22k', 224, 'beitv', 303.4),
    ('caformer_s18', 'caformer_s18.sail_in22k_ft_in1k', 224, 'caformer', 23.2),
    ('caformer_m36', 'caformer_m36.sail_in22k_ft_in1k', 224, 'caformer', 52.6),
    ('cait_xxs24_224', 'cait_xxs24_224.fb_dist_in1k', 224, 'cait', 11.8),
    ('cait_s36_384', 'cait_s36_384.fb_dist_in1k', 384, 'cait', 68.0),
    ('coat_tiny', 'coat_tiny.in1k', 224, 'coat', 5.3),
    ('coat_lite_medium', 'coat_lite_medium.in1k', 224, 'coat', 44.1),
    ('coatnet_nano_rw_224', 'coatnet_nano_rw_224.sw_in1k', 224, 'coatnet', 14.6),
    ('coatnet_2_rw_224', 'coatnet_2_rw_224.sw_in12k_ft_in1k', 224, 'coatnet', 72.8),
    ('coatnext_nano_rw_224', 'coatnext_nano_rw_224.sw_in1k', 224, 'coatnext', 14.2),
    ('convformer_s18', 'convformer_s18.sail_in22k_ft_in1k', 224, 'convformer', 23.7),
    ('convformer_m36', 'convformer_m36.sail_in22k_ft_in1k', 224, 'convformer', 53.4),
    ('convit_tiny', 'convit_tiny.fb_in1k', 224, 'convit', 5.5),
    ('convit_base', 'convit_base.fb_in1k', 224, 'convit', 85.8),
    ('convmixer_768_32', 'convmixer_768_32.in1k', 224, 'convmixer', 20.3),
    ('convmixer_1536_20', 'convmixer_1536_20.in1k', 224, 'convmixer', 50.1),
    ('convnext_femto', 'convnext_femto.d1_in1k', 224, 'convnext', 4.8),
    ('convnext_tiny', 'convnext_tiny.dinov3_lvd1689m', 224, 'convnext', 27.8),
    ('convnextv2_femto', 'convnextv2_femto.fcmae', 224, 'convnextv', 4.8),
    ('convnextv2_base', 'convnextv2_base.fcmae_ft_in22k_in1k', 224, 'convnextv', 87.7),
    ('cpubone_nano', 'cpubone_nano.r224_in1k', 224, 'cpubone', 4.9),
    ('cpubone_b3', 'cpubone_b3.r224_in1k', 224, 'cpubone', 39.1),
    ('crossvit_tiny_240', 'crossvit_tiny_240.in1k', 240, 'crossvit', 6.7),
    ('crossvit_18_dagger_408', 'crossvit_18_dagger_408.in1k', 408, 'crossvit', 43.9),
    ('cs3darknet_focus_m', 'cs3darknet_focus_m.c2ns_in1k', 256, 'cs', 8.5),
    ('cs3se_edgenet_x', 'cs3se_edgenet_x.c2ns_in1k', 256, 'cs', 49.4),
    ('csatv2', 'csatv2.r512_in1k', 512, 'csatv', 10.7),
    ('csatv2_21m', 'csatv2_21m.sw_r512_in1k', 512, 'csatv', 20.2),
    ('cspdarknet53', 'cspdarknet53.ra_in1k', 256, 'cspdarknet', 26.6),
    ('cspresnet50', 'cspresnet50.ra_in1k', 256, 'cspresnet', 20.6),
    ('cspresnext50', 'cspresnext50.ra_in1k', 256, 'cspresnext', 18.5),
    ('darknet53', 'darknet53.c2ns_in1k', 256, 'darknet', 40.6),
    ('darknetaa53', 'darknetaa53.c2ns_in1k', 256, 'darknetaa', 35.0),
    ('davit_tiny', 'davit_tiny.msft_in1k', 224, 'davit', 27.6),
    ('davit_small', 'davit_small.msft_in1k', 224, 'davit', 49.0),
    ('deit_tiny_distilled_patch16_224', 'deit_tiny_distilled_patch16_224.fb_in1k', 224, 'deit', 5.5),
    ('deit3_base_patch16_224', 'deit3_base_patch16_224.fb_in1k', 224, 'deit', 85.8),
    ('densenet121', 'densenet121.ra_in1k', 224, 'densenet', 7.0),
    ('densenet161', 'densenet161.tv_in1k', 224, 'densenet', 26.5),
    ('densenetblur121d', 'densenetblur121d.ra_in1k', 224, 'densenetblur', 7.0),
    ('dla34', 'dla34.in1k', 224, 'dla', 15.2),
    ('dla169', 'dla169.in1k', 224, 'dla', 52.4),
    ('dla102x2', 'dla102x2.in1k', 224, 'dla102x', 40.3),
    ('dm_nfnet_f0', 'dm_nfnet_f0.dm_in1k', 224, 'dm', 68.4),
    ('dm_nfnet_f1', 'dm_nfnet_f1.dm_in1k', 224, 'dm', 129.6),
    ('dpn68', 'dpn68.mx_in1k', 224, 'dpn', 11.8),
    ('dpn98', 'dpn98.mx_in1k', 224, 'dpn', 58.9),
    ('eca_resnext26ts', 'eca_resnext26ts.ch_in1k', 256, 'eca', 8.2),
    ('eca_nfnet_l2', 'eca_nfnet_l2.ra3_in1k', 320, 'eca', 53.6),
    ('ecaresnet26t', 'ecaresnet26t.ra2_in1k', 256, 'ecaresnet', 14.0),
    ('ecaresnet101d', 'ecaresnet101d.miil_in1k', 224, 'ecaresnet', 42.5),
    ('ecaresnetlight', 'ecaresnetlight.miil_in1k', 224, 'ecaresnetlight', 28.1),
    ('edgenext_small', 'edgenext_small.usi_in1k', 256, 'edgenext', 5.3),
    ('edgenext_base', 'edgenext_base.in21k_ft_in1k', 256, 'edgenext', 17.9),
    ('efficientformer_l1', 'efficientformer_l1.snap_dist_in1k', 224, 'efficientformer', 11.4),
    ('efficientformer_l7', 'efficientformer_l7.snap_dist_in1k', 224, 'efficientformer', 80.7),
    ('efficientformerv2_s1', 'efficientformerv2_s1.snap_dist_in1k', 224, 'efficientformerv', 5.7),
    ('efficientformerv2_l', 'efficientformerv2_l.snap_dist_in1k', 224, 'efficientformerv', 25.6),
    ('efficientnet_b0', 'efficientnet_b0.ra4_e3600_r224_in1k', 224, 'efficientnet', 4.0),
    ('efficientnet_h_b5', 'efficientnet_h_b5.sw_r448_e450_in1k', 448, 'efficientnet', 43.4),
    ('efficientnetv2_rw_t', 'efficientnetv2_rw_t.ra2_in1k', 224, 'efficientnetv', 12.6),
    ('efficientnetv2_rw_m', 'efficientnetv2_rw_m.agc_in1k', 320, 'efficientnetv', 51.1),
    ('efficientvim_m1', 'efficientvim_m1.e300_in1k', 224, 'efficientvim', 5.7),
    ('efficientvim_m4', 'efficientvim_m4.e300_in1k', 256, 'efficientvim', 18.0),
    ('efficientvit_m2', 'efficientvit_m2.r224_in1k', 224, 'efficientvit', 4.0),
    ('efficientvit_l2', 'efficientvit_l2.r224_in1k', 224, 'efficientvit', 60.5),
    ('ese_vovnet19b_dw', 'ese_vovnet19b_dw.ra_in1k', 224, 'ese', 5.5),
    ('ese_vovnet57b', 'ese_vovnet57b.ra4_e3600_r256_in1k', 256, 'ese', 37.6),
    ('eva02_tiny_patch14_224', 'eva02_tiny_patch14_224.mim_in22k', 224, 'eva', 5.5),
    ('eva02_base_patch14_224', 'eva02_base_patch14_224.mim_in22k', 224, 'eva', 85.8),
    ('eva02_large_patch14_clip_224', 'eva02_large_patch14_clip_224.merged2b', 224, 'eva_clip', 303.3),
    ('eva02_large_patch14_clip_336', 'eva02_large_patch14_clip_336.merged2b', 336, 'eva_clip', 303.6),
    ('fasternet_t1', 'fasternet_t1.in1k', 224, 'fasternet', 6.3),
    ('fasternet_m', 'fasternet_m.in1k', 224, 'fasternet', 52.2),
    ('fastvit_t12', 'fastvit_t12.apple_dist_in1k', 256, 'fastvit', 6.5),
    ('fastvit_ma36', 'fastvit_ma36.apple_dist_in1k', 256, 'fastvit', 42.9),
    ('fbnetv3_b', 'fbnetv3_b.ra2_in1k', 224, 'fbnetv', 6.6),
    ('fbnetv3_g', 'fbnetv3_g.ra2_in1k', 240, 'fbnetv', 14.6),
    ('flexivit_small', 'flexivit_small.1200ep_in1k', 240, 'flexivit', 21.7),
    ('flexivit_base', 'flexivit_base.1000ep_in21k', 240, 'flexivit', 85.8),
    ('focalnet_tiny_srf', 'focalnet_tiny_srf.ms_in1k', 224, 'focalnet', 27.7),
    ('focalnet_small_lrf', 'focalnet_small_lrf.ms_in1k', 224, 'focalnet', 49.6),
    ('gc_efficientnetv2_rw_t', 'gc_efficientnetv2_rw_t.agc_in1k', 224, 'gc', 12.7),
    ('gcresnet33ts', 'gcresnet33ts.ra2_in1k', 256, 'gcresnet', 18.6),
    ('gcresnet50t', 'gcresnet50t.ra2_in1k', 256, 'gcresnet', 23.8),
    ('gcresnext26ts', 'gcresnext26ts.ch_in1k', 256, 'gcresnext', 8.4),
    ('gcresnext50ts', 'gcresnext50ts.ch_in1k', 256, 'gcresnext', 13.6),
    ('gcvit_xxtiny', 'gcvit_xxtiny.in1k', 224, 'gcvit', 11.5),
    ('gcvit_small', 'gcvit_small.in1k', 224, 'gcvit', 50.3),
    ('gernet_s', 'gernet_s.idstcv_in1k', 224, 'gernet', 6.3),
    ('gernet_l', 'gernet_l.idstcv_in1k', 256, 'gernet', 28.5),
    ('ghostnetv2_100', 'ghostnetv2_100.in1k', 224, 'ghostnetv', 4.9),
    ('ghostnetv2_160', 'ghostnetv2_160.in1k', 224, 'ghostnetv', 11.1),
    ('hardcorenas_a', 'hardcorenas_a.miil_green_in1k', 224, 'hardcorenas', 4.0),
    ('hardcorenas_f', 'hardcorenas_f.miil_green_in1k', 224, 'hardcorenas', 6.9),
    ('hgnet_tiny', 'hgnet_tiny.paddle_in1k', 224, 'hgnet', 12.7),
    ('hgnet_base', 'hgnet_base.ssld_in1k', 224, 'hgnet', 69.5),
    ('hgnetv2_b1', 'hgnetv2_b1.ssld_stage1_in22k_in1k', 224, 'hgnetv', 4.3),
    ('hgnetv2_b6', 'hgnetv2_b6.ssld_stage1_in22k_in1k', 224, 'hgnetv', 73.2),
    ('hiera_tiny_224', 'hiera_tiny_224.mae', 224, 'hiera', 27.1),
    ('hiera_base_plus_224', 'hiera_base_plus_224.mae', 224, 'hiera', 69.0),
    ('hrnet_w18_small', 'hrnet_w18_small.gluon_in1k', 224, 'hrnet', 11.1),
    ('hrnet_w40', 'hrnet_w40.ms_in1k', 224, 'hrnet', 55.5),
    ('iformer_s', 'iformer_s.in1k', 224, 'iformer', 6.2),
    ('iformer_h', 'iformer_h.in1k', 224, 'iformer', 98.8),
    ('inception_v3', 'inception_v3.gluon_in1k', 299, 'inception', 21.8),
    ('inception_resnet_v2', 'inception_resnet_v2.tf_ens_adv_in1k', 299, 'inception', 54.3),
    ('lcnetv2_base', 'lcnetv2_base.paddle_in1k', 224, 'lcnetv', 5.4),
    ('lcnetv2_large', 'lcnetv2_large.paddle_in1k', 224, 'lcnetv', 9.3),
    ('levit_128s', 'levit_128s.fb_dist_in1k', 224, 'levit', 7.0),
    ('levit_384', 'levit_384.fb_dist_in1k', 224, 'levit', 37.6),
    ('lowformer_b0', 'lowformer_b0.in1k', 224, 'lowformer', 12.5),
    ('lowformer_b3', 'lowformer_b3.in1k', 224, 'lowformer', 55.5),
    ('mambaout_femto', 'mambaout_femto.in1k', 224, 'mambaout', 6.2),
    ('mambaout_small', 'mambaout_small.in1k', 224, 'mambaout', 46.2),
    ('maxvit_rmlp_pico_rw_256', 'maxvit_rmlp_pico_rw_256.sw_in1k', 256, 'maxvit', 7.3),
    ('maxvit_rmlp_small_rw_224', 'maxvit_rmlp_small_rw_224.sw_in1k', 224, 'maxvit', 64.1),
    ('maxxvit_rmlp_nano_rw_256', 'maxxvit_rmlp_nano_rw_256.sw_in1k', 256, 'maxxvit', 16.3),
    ('maxxvit_rmlp_small_rw_256', 'maxxvit_rmlp_small_rw_256.sw_in1k', 256, 'maxxvit', 65.2),
    ('maxxvitv2_nano_rw_256', 'maxxvitv2_nano_rw_256.sw_in1k', 256, 'maxxvitv', 22.9),
    ('maxxvitv2_rmlp_base_rw_224', 'maxxvitv2_rmlp_base_rw_224.sw_in12k_ft_in1k', 224, 'maxxvitv', 115.1),
    ('mixnet_l', 'mixnet_l.ft_in1k', 224, 'mixnet', 5.8),
    ('mixnet_xl', 'mixnet_xl.ra_in1k', 224, 'mixnet', 10.4),
    ('mobilenet_edgetpu_v2_m', 'mobilenet_edgetpu_v2_m.ra4_e3600_r224_in1k', 224, 'mobilenet', 7.1),
    ('mobilenetv3_large_100', 'mobilenetv3_large_100.miil_in21k_ft_in1k', 224, 'mobilenetv', 4.2),
    ('mobilenetv4_hybrid_large', 'mobilenetv4_hybrid_large.e600_r384_in1k', 384, 'mobilenetv', 36.5),
    ('mobileone_s0', 'mobileone_s0.apple_in1k', 224, 'mobileone', 4.3),
    ('mobileone_s4', 'mobileone_s4.apple_in1k', 224, 'mobileone', 12.9),
    ('mobilevit_s', 'mobilevit_s.cvnets_in1k', 256, 'mobilevit', 4.9),
    ('mobilevitv2_100', 'mobilevitv2_100.cvnets_in1k', 256, 'mobilevitv', 4.4),
    ('mobilevitv2_200', 'mobilevitv2_200.cvnets_in22k_ft_in1k', 256, 'mobilevitv', 17.4),
    ('mvitv2_tiny', 'mvitv2_tiny.fb_in1k', 224, 'mvitv', 23.4),
    ('mvitv2_base', 'mvitv2_base.fb_in1k', 224, 'mvitv', 50.7),
    ('nest_tiny_jx', 'nest_tiny_jx.goog_in1k', 224, 'nest', 16.7),
    ('nest_base_jx', 'nest_base_jx.goog_in1k', 224, 'nest', 67.2),
    ('nextvit_small', 'nextvit_small.bd_in1k', 224, 'nextvit', 30.7),
    ('nextvit_large', 'nextvit_large.bd_in1k', 224, 'nextvit', 56.8),
    ('nf_regnet_b1', 'nf_regnet_b1.ra2_in1k', 256, 'nf', 9.3),
    ('nf_resnet50', 'nf_resnet50.ra2_in1k', 256, 'nf', 23.5),
    ('nfnet_l0', 'nfnet_l0.ra2_in1k', 224, 'nfnet', 32.8),
    ('pit_ti_224', 'pit_ti_224.in1k', 224, 'pit', 4.6),
    ('pit_b_224', 'pit_b_224.in1k', 224, 'pit', 72.7),
    ('poolformer_s12', 'poolformer_s12.sail_in1k', 224, 'poolformer', 11.4),
    ('poolformer_m36', 'poolformer_m36.sail_in1k', 224, 'poolformer', 55.4),
    ('poolformerv2_s12', 'poolformerv2_s12.sail_in1k', 224, 'poolformerv', 11.4),
    ('poolformerv2_m36', 'poolformerv2_m36.sail_in1k', 224, 'poolformerv', 55.3),
    ('pvt_v2_b1', 'pvt_v2_b1.in1k', 224, 'pvt', 13.5),
    ('pvt_v2_b4', 'pvt_v2_b4.in1k', 224, 'pvt', 62.0),
    ('rdnet_tiny', 'rdnet_tiny.nv_in1k', 224, 'rdnet', 22.8),
    ('rdnet_small', 'rdnet_small.nv_in1k', 224, 'rdnet', 49.2),
    ('regnetv_040', 'regnetv_040.ra3_in1k', 224, 'regnetv', 19.6),
    ('regnetv_064', 'regnetv_064.ra3_in1k', 224, 'regnetv', 29.3),
    ('regnetx_004', 'regnetx_004.pycls_in1k', 224, 'regnetx', 4.8),
    ('regnetx_160', 'regnetx_160.pycls_in1k', 224, 'regnetx', 52.2),
    ('regnety_006', 'regnety_006.pycls_in1k', 224, 'regnety', 5.4),
    ('regnety_120', 'regnety_120.sw_in12k_ft_in1k', 224, 'regnety', 49.6),
    ('regnetz_b16', 'regnetz_b16.ra3_in1k', 224, 'regnetz', 8.2),
    ('regnetz_e8', 'regnetz_e8.ra3_in1k', 256, 'regnetz', 55.6),
    ('repghostnet_130', 'repghostnet_130.in1k', 224, 'repghostnet', 4.2),
    ('repghostnet_200', 'repghostnet_200.in1k', 224, 'repghostnet', 8.5),
    ('repvit_m0_9', 'repvit_m0_9.dist_300e_in1k', 224, 'repvit', 4.7),
    ('repvit_m2_3', 'repvit_m2_3.dist_300e_in1k', 224, 'repvit', 22.4),
    ('res2net50_14w_8s', 'res2net50_14w_8s.in1k', 224, 'res2net', 23.0),
    ('res2net50_26w_8s', 'res2net50_26w_8s.in1k', 224, 'res2net', 46.4),
    ('res2next50', 'res2next50.in1k', 224, 'res2next', 22.6),
    ('resnest14d', 'resnest14d.gluon_in1k', 224, 'resnest', 8.6),
    ('resnest200e', 'resnest200e.in1k', 320, 'resnest', 68.2),
    ('resnet10t', 'resnet10t.c3_in1k', 224, 'resnet', 4.9),
    ('resnet152s', 'resnet152s.gluon_in1k', 224, 'resnet', 58.3),
    ('resnet50x4_clip', 'resnet50x4_clip.openai', 288, 'resnet50x_clip', 85.5),
    ('resnet50x16_clip', 'resnet50x16_clip.openai', 384, 'resnet50x_clip', 165.0),
    ('resnet50_clip', 'resnet50_clip.openai', 224, 'resnet_clip', 36.2),
    ('resnet101_clip', 'resnet101_clip.openai', 224, 'resnet_clip', 55.2),
    ('resnetaa50', 'resnetaa50.a1h_in1k', 224, 'resnetaa', 23.5),
    ('resnetaa101d', 'resnetaa101d.sw_in12k_ft_in1k', 224, 'resnetaa', 42.5),
    ('resnetblur50', 'resnetblur50.bt_in1k', 224, 'resnetblur', 23.5),
    ('resnetrs50', 'resnetrs50.tf_in1k', 224, 'resnetrs', 33.6),
    ('resnetrs101', 'resnetrs101.tf_in1k', 224, 'resnetrs', 61.6),
    ('resnetv2_18', 'resnetv2_18.ra4_e3600_r224_in1k', 224, 'resnetv', 11.2),
    ('resnetv2_101', 'resnetv2_101.a1h_in1k', 224, 'resnetv', 42.5),
    ('resnext26ts', 'resnext26ts.ra2_in1k', 256, 'resnext', 8.2),
    ('resnext101_64x4d', 'resnext101_64x4d.c1_in1k', 224, 'resnext', 81.4),
    ('rexnet_130', 'rexnet_130.nav_in1k', 224, 'rexnet', 5.9),
    ('rexnet_300', 'rexnet_300.nav_in1k', 224, 'rexnet', 30.9),
    ('rexnetr_200', 'rexnetr_200.sw_in12k_ft_in1k', 224, 'rexnetr', 14.0),
    ('rexnetr_300', 'rexnetr_300.sw_in12k_ft_in1k', 224, 'rexnetr', 31.0),
    ('sebotnet33ts_256', 'sebotnet33ts_256.a1h_in1k', 256, 'sebotnet', 12.4),
    ('sehalonet33ts', 'sehalonet33ts.ra2_in1k', 256, 'sehalonet', 12.4),
    ('selecsls60', 'selecsls60.in1k', 224, 'selecsls', 29.4),
    ('selecsls60b', 'selecsls60b.in1k', 224, 'selecsls', 31.7),
    ('senet154', 'senet154.gluon_in1k', 224, 'senet', 113.0),
    ('sequencer2d_s', 'sequencer2d_s.in1k', 224, 'sequencer', 27.3),
    ('sequencer2d_l', 'sequencer2d_l.in1k', 224, 'sequencer', 53.9),
    ('seresnet33ts', 'seresnet33ts.ra2_in1k', 256, 'seresnet', 18.5),
    ('seresnet152d', 'seresnet152d.ra2_in1k', 256, 'seresnet', 64.8),
    ('seresnext26ts', 'seresnext26ts.ch_in1k', 256, 'seresnext', 8.3),
    ('seresnext101_32x4d', 'seresnext101_32x4d.gluon_in1k', 224, 'seresnext', 46.9),
    ('seresnextaa101d_32x8d', 'seresnextaa101d_32x8d.sw_in12k_ft_in1k', 224, 'seresnextaa', 91.5),
    ('seresnextaa201d_32x8d', 'seresnextaa201d_32x8d.sw_in12k_ft_in1k_384', 384, 'seresnextaa', 147.3),
    ('shvit_s1', 'shvit_s1.in1k', 224, 'shvit', 6.0),
    ('shvit_s4', 'shvit_s4.in1k', 256, 'shvit', 16.1),
    ('skresnet18', 'skresnet18.ra_in1k', 224, 'skresnet', 11.4),
    ('skresnet34', 'skresnet34.ra_in1k', 224, 'skresnet', 21.8),
    ('skresnext50_32x4d', 'skresnext50_32x4d.ra_in1k', 224, 'skresnext', 25.4),
    ('starnet_s3', 'starnet_s3.in1k', 224, 'starnet', 5.5),
    ('starnet_s4', 'starnet_s4.in1k', 224, 'starnet', 7.2),
    ('swiftformer_s', 'swiftformer_s.dist_in1k', 224, 'swiftformer', 5.6),
    ('swiftformer_l3', 'swiftformer_l3.dist_in1k', 224, 'swiftformer', 27.5),
    ('swin_tiny_patch4_window7_224', 'swin_tiny_patch4_window7_224.ms_in22k_ft_in1k', 224, 'swin', 27.5),
    ('swin_s3_base_224', 'swin_s3_base_224.ms_in1k', 224, 'swin', 70.4),
    ('swinv2_cr_tiny_ns_224', 'swinv2_cr_tiny_ns_224.sw_in1k', 224, 'swinv', 27.6),
    ('swinv2_small_window8_256', 'swinv2_small_window8_256.ms_in1k', 256, 'swinv', 49.0),
    ('tf_efficientnet_b0', 'tf_efficientnet_b0.aa_in1k', 224, 'tf', 4.0),
    ('tf_efficientnet_b7', 'tf_efficientnet_b7.aa_in1k', 512, 'tf', 63.8),
    ('tinynet_a', 'tinynet_a.in1k', 224, 'tinynet', 4.9),
    ('tnt_s_legacy_patch16_224', 'tnt_s_legacy_patch16_224.in1k', 224, 'tnt', 23.4),
    ('tnt_b_patch16_224', 'tnt_b_patch16_224.in1k', 224, 'tnt', 64.8),
    ('tresnet_m', 'tresnet_m.miil_in21k_ft_in1k', 224, 'tresnet', 29.3),
    ('tresnet_l', 'tresnet_l.miil_in1k', 224, 'tresnet', 53.6),
    ('twins_svt_small', 'twins_svt_small.in1k', 224, 'twins', 23.5),
    ('twins_pcpvt_large', 'twins_pcpvt_large.in1k', 224, 'twins', 60.5),
    ('visformer_tiny', 'visformer_tiny.in1k', 224, 'visformer', 9.9),
    ('visformer_small', 'visformer_small.in1k', 224, 'visformer', 39.5),
    ('vit_tiny_patch16_224', 'vit_tiny_patch16_224.augreg_in21k_ft_in1k', 224, 'vit', 5.5),
    ('vit_tiny_patch16_384', 'vit_tiny_patch16_384.augreg_in21k_ft_in1k', 384, 'vit', 5.6),
    ('vit_tiny_r_s16_p8_224', 'vit_tiny_r_s16_p8_224.augreg_in21k_ft_in1k', 224, 'vit', 6.1),
    ('vit_tiny_r_s16_p8_384', 'vit_tiny_r_s16_p8_384.augreg_in21k_ft_in1k', 384, 'vit', 6.2),
    ('vit_dwee_patch16_reg1_gap_256', 'vit_dwee_patch16_reg1_gap_256.sbb_in1k', 256, 'vit', 13.2),
    ('vit_wee_patch16_reg1_gap_256', 'vit_wee_patch16_reg1_gap_256.sbb_in1k', 256, 'vit', 13.2),
    ('vit_dpwee_patch16_reg1_gap_256', 'vit_dpwee_patch16_reg1_gap_256.sbb_in1k', 256, 'vit', 15.0),
    ('vit_pwee_patch16_reg1_gap_256', 'vit_pwee_patch16_reg1_gap_256.sbb_in1k', 256, 'vit', 15.0),
    ('vit_relpos_small_patch16_224', 'vit_relpos_small_patch16_224.sw_in1k', 224, 'vit', 21.6),
    ('vit_srelpos_small_patch16_224', 'vit_srelpos_small_patch16_224.sw_in1k', 224, 'vit', 21.6),
    ('vit_small_patch8_224', 'vit_small_patch8_224.dino', 224, 'vit', 21.7),
    ('vit_small_patch16_224', 'vit_small_patch16_224.augreg_in21k_ft_in1k', 224, 'vit', 21.7),
    ('vit_small_patch16_384', 'vit_small_patch16_384.augreg_in21k_ft_in1k', 384, 'vit', 21.8),
    ('vit_dlittle_patch16_reg1_gap_256', 'vit_dlittle_patch16_reg1_gap_256.sbb_nadamuon_in1k', 256, 'vit', 22.2),
    ('vit_little_patch16_reg1_gap_256', 'vit_little_patch16_reg1_gap_256.sbb_in12k_ft_in1k', 256, 'vit', 22.2),
    ('vit_little_patch16_reg4_gap_256', 'vit_little_patch16_reg4_gap_256.sbb_in1k', 256, 'vit', 22.2),
    ('vit_small_patch32_224', 'vit_small_patch32_224.augreg_in21k_ft_in1k', 224, 'vit', 22.5),
    ('vit_small_patch32_384', 'vit_small_patch32_384.augreg_in21k_ft_in1k', 384, 'vit', 22.5),
    ('vit_small_r26_s32_224', 'vit_small_r26_s32_224.augreg_in21k_ft_in1k', 224, 'vit', 36.0),
    ('vit_relpos_medium_patch16_224', 'vit_relpos_medium_patch16_224.sw_in1k', 224, 'vit', 38.2),
    ('vit_relpos_medium_patch16_rpn_224', 'vit_relpos_medium_patch16_rpn_224.sw_in1k', 224, 'vit', 38.2),
    ('vit_srelpos_medium_patch16_224', 'vit_srelpos_medium_patch16_224.sw_in1k', 224, 'vit', 38.2),
    ('vit_relpos_medium_patch16_cls_224', 'vit_relpos_medium_patch16_cls_224.sw_in1k', 224, 'vit', 38.3),
    ('vit_medium_patch16_gap_240', 'vit_medium_patch16_gap_240.sw_in12k', 240, 'vit', 38.3),
    ('vit_medium_patch16_gap_256', 'vit_medium_patch16_gap_256.sw_in12k_ft_in1k', 256, 'vit', 38.3),
    ('vit_medium_patch16_reg1_gap_256', 'vit_medium_patch16_reg1_gap_256.sbb_in1k', 256, 'vit', 38.4),
    ('vit_medium_patch16_reg4_gap_256', 'vit_medium_patch16_reg4_gap_256.sbb_in12k_ft_in1k', 256, 'vit', 38.4),
    ('vit_betwixt_patch16_reg1_gap_256', 'vit_betwixt_patch16_reg1_gap_256.sbb_in1k', 256, 'vit', 59.8),
    ('vit_betwixt_patch16_reg4_gap_256', 'vit_betwixt_patch16_reg4_gap_256.sbb2_e200_in12k_ft_in1k', 256, 'vit', 59.8),
    ('vit_betwixt_patch16_reg4_gap_384', 'vit_betwixt_patch16_reg4_gap_384.sbb2_e200_in12k_ft_in1k', 384, 'vit', 60.0),
    ('vit_mediumd_patch16_reg4_gap_256', 'vit_mediumd_patch16_reg4_gap_256.sbb2_e200_in12k_ft_in1k', 256, 'vit', 63.6),
    ('vit_relpos_base_patch16_224', 'vit_relpos_base_patch16_224.sw_in1k', 224, 'vit', 85.7),
    ('vit_relpos_base_patch16_clsgap_224', 'vit_relpos_base_patch16_clsgap_224.sw_in1k', 224, 'vit', 85.7),
    ('vit_base_patch8_224', 'vit_base_patch8_224.augreg2_in21k_ft_in1k', 224, 'vit', 85.8),
    ('vit_base_patch16_224', 'vit_base_patch16_224.augreg2_in21k_ft_in1k', 224, 'vit', 85.8),
    ('vit_base_patch16_224_miil', 'vit_base_patch16_224_miil.in21k_ft_in1k', 224, 'vit', 85.8),
    ('vit_base_patch16_rpn_224', 'vit_base_patch16_rpn_224.sw_in1k', 224, 'vit', 85.8),
    ('vit_base_mci_224', 'vit_base_mci_224.apple_mclip2_dfndr2b', 224, 'vit', 86.0),
    ('vit_xsmall_patch16_clip_224', 'vit_xsmall_patch16_clip_224.tinyclip_yfcc15m', 224, 'vit_clip', 8.1),
    ('vit_betwixt_patch32_clip_224', 'vit_betwixt_patch32_clip_224.tinyclip_laion400m', 224, 'vit_clip', 61.1),
    ('vit_small_patch14_dinov2', 'vit_small_patch14_dinov2.lvd142m', 504, 'vit_dinov2', 22.1),
    ('vit_base_patch14_dinov2', 'vit_base_patch14_dinov2.lvd142m', 504, 'vit_dinov2', 86.6),
    ('vit_tiny_patch16_dinov3_qkvb', 'vit_tiny_patch16_dinov3_qkvb.eupe_lvd1689m', 256, 'vit_dinov3', 5.5),
    ('vit_base_patch16_dinov3_qkvb', 'vit_base_patch16_dinov3_qkvb.eupe_lvd1689m', 256, 'vit_dinov3', 85.7),
    ('vit_pe_core_tiny_patch16_384', 'vit_pe_core_tiny_patch16_384.fb', 384, 'vit_pe_core', 6.0),
    ('vit_pe_core_small_patch16_384', 'vit_pe_core_small_patch16_384.fb', 384, 'vit_pe_core', 23.6),
    ('vit_pe_lang_large_patch14_448', 'vit_pe_lang_large_patch14_448.fb', 448, 'vit_pe_lang', 291.4),
    ('vit_pe_spatial_tiny_patch16_512', 'vit_pe_spatial_tiny_patch16_512.fb', 512, 'vit_pe_spatial', 5.7),
    ('vit_pe_spatial_small_patch16_512', 'vit_pe_spatial_small_patch16_512.fb', 512, 'vit_pe_spatial', 22.0),
    ('vit_small_patch16_rope_224', 'vit_small_patch16_rope_224.naver_in1k', 224, 'vit_rope', 21.6),
    ('vit_betwixt_patch16_rope_reg4_gap_256', 'vit_betwixt_patch16_rope_reg4_gap_256.sbb_in1k', 256, 'vit_rope', 59.6),
    ('vit_base_patch32_siglip_256', 'vit_base_patch32_siglip_256.v2_webli', 256, 'vit_siglip', 94.6),
    ('vit_so400m_patch14_siglip_224', 'vit_so400m_patch14_siglip_224.v2_webli', 224, 'vit_siglip', 427.7),
    ('vitamin_small_224', 'vitamin_small_224.datacomp1b_clip', 224, 'vitamin', 21.9),
    ('vitamin_base_224', 'vitamin_base_224.datacomp1b_clip', 224, 'vitamin', 87.1),
    ('volo_d1_224', 'volo_d1_224.sail_in1k', 224, 'volo', 25.9),
    ('volo_d2_384', 'volo_d2_384.sail_in1k', 384, 'volo', 57.8),
    ('wide_resnet50_2', 'wide_resnet50_2.racm_in1k', 224, 'wide', 66.8),
    ('wide_resnet101_2', 'wide_resnet101_2.tv2_in1k', 224, 'wide', 124.8),
    ('xception41', 'xception41.tf_in1k', 299, 'xception', 24.9),
    ('xception71', 'xception71.tf_in1k', 299, 'xception', 40.3),
    ('xcit_tiny_12_p8_224', 'xcit_tiny_12_p8_224.fb_dist_in1k', 224, 'xcit', 6.5),
    ('xcit_small_24_p16_224', 'xcit_small_24_p16_224.fb_dist_in1k', 224, 'xcit', 47.3),
]
_sudah = {c["name"] for c in CANDIDATES}
for _tag, _nm, _h, _fam, _pm in KATALOG:
    if _nm in _sudah: continue
    _mult = max(_pm / 86.0, 0.15)
    CANDIDATES.append(K(_tag, _nm, _h, _fam, 6 if _mult * (_h / 224) ** 2 <= 1.5 else 7,
                        round(_mult, 2)))

# INFEST_KANDIDAT=kurasi membatasi ke 43 kandidat pilihan tangan saja (sekitar 4 jam).
if os.environ.get("INFEST_KANDIDAT", "semua") == "kurasi":
    CANDIDATES = [c for c in CANDIDATES if c["prio"] <= 5]

# -----------------------------------------------------------------------------
# PROBE, PENALAAN, DAN PEMBANDING
# -----------------------------------------------------------------------------
# Kisi penalaan. v1 menemukan semua lima teratas memilih C=16 tanpa L2, yaitu
# di TEPI kisinya (C terbesar yang dicoba). Titik terbaik yang jatuh di tepi
# kisi adalah tanda kisinya terlalu sempit, jadi v2 memperluas ke C=256.
GRID = dict(C=(1.0, 4.0, 16.0, 64.0, 256.0), norm=("l2", "std", "raw"),
            pool=("bawaan", "alt"))
# Kisi tingkat pertama, untuk SEMUA kandidat. Kisi penuh di atas hanya untuk
# TOP_PENUH kandidat teratas setelah tingkat pertama (lihat Bagian 4.2).
GRID_CEPAT = dict(C=(4.0, 16.0, 64.0), norm=("l2", "raw"), pool=("bawaan", "alt"))
TOP_PENUH  = 40
TOP_GAMBAR = 50
# Pembagian fold untuk pengukuran ulang. Seed 42 identik dengan v12 sampai v17,
# jadi angka di seed itu sebanding langsung dengan sejarah proyek. Setelan
# terbaik dipilih DI seed 42, lalu diukur tanpa diubah di dua seed lain. Selisih
# antara keduanya adalah besarnya bias karena memilih dari 30 setelan.
SEEDS_ULANG = (7, 2026)
KNN_K   = (5, 10, 20)
MLP     = dict(enable=True, top=15, hidden=512, drop=0.3, epochs=120, lr=1e-3, wd=1e-2)
BOOT    = dict(n=1000, seed=0)
ENS     = dict(max_k=5, top=14)
DIV_TOP = 12
# Ambang derau. Tiga seed fine-tune v13 di fold 0 memberi simpangan baku 0,0015.
# Selisih yang lebih kecil dari ini tidak diperlakukan sebagai temuan.
NOISE   = 0.0015

# Konfirmasi fine-tune. Tujuannya BUKAN mengalahkan v15, tetapi menguji apakah
# urutan probe bertahan saat backbone-nya ikut dilatih. Semua kandidat dapat
# anggaran yang sama persis: fold 0, jumlah epoch sama, resep sama.
CONFIRM = dict(enable=True, k=4, epochs=6, fold=0, lr_body=3e-5, lr_head=1e-3,
               layer_decay=0.75, max_params_m=130, anchor="siglip2-384")

CFG = dict(n_folds=5, batch=32, nw=min(4, os.cpu_count() or 2), wd=0.05, ls=0.05)
CACHE_MAX = 1280
AR_SPLIT  = 3.0
REF = dict(probe_siglip2_224_v12=0.9250, ft_siglip2_224=0.9735, lb_v12=0.87749,
           lb_v15=0.90090, ft_fold0_siglip2_384=(0.9849, 0.9857, 0.9879),
           ft_fold0_dinov3=0.9858,
           v1_tuned={"siglip2-512": 0.9472, "siglip2-384": 0.9439, "dinov3-b16": 0.9361,
                     "siglip2L-384": 0.9271, "dinov3-cnx-b": 0.9188})

if SMOKE:
    # Uji jalur kode di CPU: sedikit citra, resolusi kecil, bobot acak.
    # Angka yang keluar dalam mode ini TIDAK BERARTI APA PUN.
    _keep = ("siglip2-224", "siglip2-384", "dinov3-b16", "cnx-v7-384", "pe-core-b384",
             "effv2-m384") + ('coat_tiny', 'crossvit_tiny_240', 'efficientnet_b0')
    CANDIDATES = [dict(c, h=96 if c["prio"] <= 5 else c["h"], w=96 if c["prio"] <= 5 else c["w"])
                  for c in CANDIDATES if c["tag"] in _keep]
    GRID = dict(C=(1.0, 16.0), norm=("l2", "std", "raw"), pool=("bawaan", "alt"))
    GRID_CEPAT = dict(C=(16.0,), norm=("raw",), pool=("bawaan", "alt")); TOP_PENUH = 3
    MLP.update(top=3, epochs=20); BOOT.update(n=100); CONFIRM.update(k=2, epochs=1)
    DIV_TOP = 5; ENS.update(top=5)

def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    os.environ["PYTHONHASHSEED"] = str(s)
seed_all()
DEV = "cuda" if torch.cuda.is_available() else "cpu"
T_START = time.time()
def el_h():  return (time.time() - T_START) / 3600.0
def rem_h(): return BUDGET["total_h"] - el_h()
def clk():   return f"[t={el_h():5.2f}j sisa={rem_h():5.2f}j]"

print(f"device={DEV} | torch={torch.__version__} | timm={timm.__version__}"
      + ("  [SMOKE: bobot boleh ACAK]" if SMOKE else ""))
if DEV == "cuda":
    _p = torch.cuda.get_device_properties(0)
    print(f"GPU: {_p.name} | VRAM {_p.total_memory/2**30:.1f} GB")
print(f"{len(CANDIDATES)} kandidat, {len({c['fam'] for c in CANDIDATES})} keluarga | "
      f"anggaran {BUDGET['total_h']:.1f} jam")

### 1.3 Lingkungan kerja dan penyimpanan

Notebook ini jalan di Kaggle maupun Colab. Yang membedakan keduanya adalah
penyimpanan. Di Kaggle `/kaggle/working` bertahan selama sesi dan ikut
tersimpan sebagai output. Di Colab `/content` hilang saat sesi putus, jadi
semua hasil antara ditaruh di Google Drive (`MyDrive/infest/automl2`) dan
cache citra v15 di `MyDrive/infest/_cache13` dipakai ulang kalau ada.

Hasil antara yang paling mahal adalah fitur tiap backbone. Setiap backbone
yang selesai diekstrak langsung ditulis ke `automl2/_fitur/`, sehingga
menjalankan ulang notebook setelah sesi putus hanya membayar backbone yang
belum selesai. Itulah satu-satunya checkpoint yang dibutuhkan di sini.

**Melanjutkan di sesi Kaggle berikutnya.** Kalau ringkasan di akhir menyebut
ada kandidat yang belum diekstrak, lakukan tiga langkah. Pertama, pastikan sesi
sebelumnya disimpan sebagai versi (Save Version), supaya folder
`automl2/_fitur` dan `automl2/_tala` ikut tersimpan sebagai output. Kedua, di
notebook yang sama klik Add Input, pilih tab Notebook Output, lalu tambahkan
output versi tadi. Ketiga, jalankan ulang dari atas. Notebook mencari folder
`_fitur` dan `_tala` di seluruh `/kaggle/input`, melewati backbone yang sudah
selesai, dan hanya mengerjakan sisanya. Fine-tune pendek sengaja ditunda ke
sesi yang menyelesaikan seluruh ekstraksi, karena pilihan wakil keluarganya
baru pasti setelah semua kandidat terukur.

Ukuran fitur untuk 333 backbone sekitar 6 GB (disimpan float16). Itu muat di
`/kaggle/working` (batas 20 GB) dan di Google Drive gratis (15 GB), tetapi
pastikan Drive Anda masih punya ruang sebanyak itu kalau memakai Colab. Bobot
setiap backbone dihapus dari disk begitu fiturnya tersimpan, jadi unduhan
ratusan bobot tidak menumpuk.

In [ ]:
IS_KAGGLE = os.path.isdir("/kaggle/working")
IS_COLAB  = (not IS_KAGGLE) and ("google.colab" in sys.modules)
SEARCH_EXTRA = []
if IS_KAGGLE:
    WORK = "/kaggle/working"; CDIR = os.path.join(WORK, "_cache13")
    print("lingkungan: KAGGLE. Pastikan Accelerator = GPU T4 dan Internet = On.")
elif IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    _root = "/content/drive/MyDrive/infest"
    WORK = _root; CDIR = os.path.join(_root, "_cache13")
    SEARCH_EXTRA = [os.path.join(_root, "data")]
    print("lingkungan: COLAB. Hasil disimpan di Google Drive, aman dari sesi putus.")
else:
    WORK = os.environ.get("INFEST_WORK", "."); CDIR = os.path.join(WORK, "_cache13")
    print("lingkungan: lokal atau lainnya.")
OUT  = os.path.join(WORK, "automl2")
FDIR = os.path.join(OUT, "_fitur")
GDIR = os.path.join(OUT, "gambar")
for _d in (OUT, FDIR, GDIR, CDIR): os.makedirs(_d, exist_ok=True)
print(f"hasil  : {OUT}\nfitur  : {FDIR}\ngambar : {GDIR}\ncache  : {CDIR}")

def simpan_gambar(fig, nama):
    try: fig.savefig(os.path.join(GDIR, nama + ".png"), bbox_inches="tight", dpi=130)
    except Exception as e: print(f"  ! gagal menyimpan {nama}: {type(e).__name__}")

### 1.4 Memuat data

Ada jebakan di ruang kerja lomba ini: terdapat dua versi himpunan uji. Folder
`test/` berisi 1.220 citra dan cocok dengan `sample_submission.csv`, sedangkan
`test.csv` adalah daftar lama yang irisannya dengan `sample_submission.csv`
nol. Daftar citra uji selalu diambil dari `sample_submission.csv`.

Pembagian fold memakai `StratifiedKFold(5, shuffle=True, random_state=42)`,
sama persis dengan v12 sampai v17. Ini bukan kebiasaan, melainkan syarat:
tanpa fold yang sama, angka di sini tidak bisa dibandingkan dengan angka
versi 1 maupun dengan OOF fine-tune yang sudah ada.

In [ ]:
SEARCH = [p for p in (SEARCH_EXTRA + ["/kaggle/input", "/kaggle/working", ".", "./data",
                                      "/content"]) if os.path.isdir(p)]
_seen = set(); SEARCH = [p for p in SEARCH if not (p in _seen or _seen.add(p))]

def find_csv(names):
    for root in SEARCH:
        for dp, dn, fs in os.walk(root, followlinks=True):
            dn[:] = [d for d in dn if not d.startswith(("_cache", "_fitur", "automl2"))]
            for f in fs:
                if f.lower() in names: return os.path.join(dp, f)
    return None

train_csv = find_csv({"train.csv"}); sub_csv = find_csv({"sample_submission.csv"})
assert train_csv, f"train.csv tidak ketemu di {SEARCH}"
assert sub_csv, "sample_submission.csv tidak ketemu. Jangan pakai test.csv: isinya daftar lama."
tr_df = pd.read_csv(train_csv)
te_df = pd.DataFrame({"image_id": pd.read_csv(sub_csv).image_id.tolist()})

for root in list(SEARCH):
    for z in glob.glob(os.path.join(root, "*.zip")):
        tag = os.path.splitext(os.path.basename(z))[0].lower()
        if tag in ("train", "test", "images"):
            out = os.path.join(WORK, "_imgs")
            if not os.path.isdir(os.path.join(out, tag)):
                os.makedirs(out, exist_ok=True)
                try:
                    with zipfile.ZipFile(z) as zf:
                        zf.extractall(out, members=[m for m in zf.namelist()
                                                    if not m.startswith("__MACOSX")])
                except Exception as e: print(f"  ! gagal ekstrak {z}: {type(e).__name__}")
if os.path.isdir(os.path.join(WORK, "_imgs")): SEARCH.append(os.path.join(WORK, "_imgs"))

EXT = (".png", ".jpg", ".jpeg", ".webp", ".bmp", ".gif", ".tif", ".tiff")
IDX, STEM = {}, {}
for root in SEARCH:
    for dp, dn, fs in os.walk(root, followlinks=True):
        dn[:] = [d for d in dn if not d.startswith(("_cache", "_fitur", "automl2"))]
        for f in fs:
            if f.lower().endswith(EXT):
                full = os.path.join(dp, f)
                IDX.setdefault(f, []).append(full)
                STEM.setdefault(os.path.splitext(f)[0], []).append(full)

def resolve(name, split):
    b = os.path.basename(str(name))
    c = IDX.get(b) or STEM.get(os.path.splitext(b)[0])
    if not c: return None
    if len(c) == 1: return c[0]
    pref = [p for p in c if f"{os.sep}{split}" in p.lower()]
    return (pref or c)[0]

tr_df["path"] = [resolve(x, "train") for x in tr_df.image_id]
te_df["path"] = [resolve(x, "test") for x in te_df.image_id]
if te_df.path.isna().any() or tr_df.path.isna().all():
    raise AssertionError(f"citra tidak ketemu (train {tr_df.path.isna().sum()}, "
                         f"test {te_df.path.isna().sum()}); folder: {SEARCH}")
tr_df = tr_df[tr_df.path.notna()].reset_index(drop=True)

if SMOKE:
    tr_df = (tr_df.sample(frac=1, random_state=0).groupby("label").head(40)
                  .reset_index(drop=True))
    te_df = te_df.sample(80, random_state=0).reset_index(drop=True)

CLASSES = sorted(tr_df.label.unique()); C2I = {c: i for i, c in enumerate(CLASSES)}
tr_df["y"] = tr_df.label.map(C2I); NC = len(CLASSES)
ytrue = tr_df.y.values; cnt = np.bincount(ytrue, minlength=NC)
print(f"train {len(tr_df)} | test {len(te_df)} | {NC} kelas: {CLASSES}")

def _ar(paths):
    out = []
    for q in paths:
        try:
            with Image.open(q) as im: out.append(im.size[0] / max(im.size[1], 1))
        except Exception: out.append(np.nan)
    return np.array(out, float)

AR_TR = _ar(tr_df.path.tolist()); AR_TE = _ar(te_df.path.tolist())
BLOK_TR = AR_TR < AR_SPLIT
W_BLOK = float((AR_TE[~np.isnan(AR_TE)] < AR_SPLIT).mean())
print(f"populasi test: blok {W_BLOK*100:.1f}%  strip {(1-W_BLOK)*100:.1f}%")
print(f"populasi train: blok {BLOK_TR.mean()*100:.1f}%  strip {(1-BLOK_TR.mean())*100:.1f}%")

def folds_for(seed):
    return list(StratifiedKFold(CFG["n_folds"], shuffle=True, random_state=seed)
                .split(np.zeros(len(ytrue)), ytrue))
FOLDS = {s: folds_for(s) for s in (SEED,) + SEEDS_ULANG}
print(f"fold: seed {SEED} (sama dengan v12 sampai v17) + seed ulang {SEEDS_ULANG}")

### 1.5 Metrik

Metrik utama di seluruh notebook ini adalah `popf1`, yaitu macro-F1 yang
ditimbang ulang menurut komposisi populasi di himpunan uji. Citra di lomba ini
terbagi dua populasi yang sangat berbeda: strip (satu baris teks, rasio aspek
3 ke atas) dan blok (beberapa baris). Di train keduanya tidak seimbang dengan
di test, sehingga macro-F1 polos memberi bobot yang salah. `popf1` menghitung
F1 terpisah di tiap populasi lalu menimbangnya dengan proporsi test.

Macro-F1 dihitung dengan `bincount` di NumPy, bukan `sklearn`, karena selang
bootstrap di Bagian 5 memanggilnya ratusan ribu kali. Hasilnya dicek sama
dengan `sklearn` di sel ini juga.

In [ ]:
def macro_f1(y, p, nc=None):
    nc = nc or NC
    cm = np.bincount(y * nc + p, minlength=nc * nc).reshape(nc, nc)
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    den = 2 * tp + fp + fn
    present = (cm.sum(1) + cm.sum(0)) > 0
    f = np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
    return float(f[present].mean()) if present.any() else 0.0

def popf1_idx(prob_or_pred, idx=None):
    """popf1 atas sebagian baris (idx), dipakai juga oleh bootstrap."""
    pred = prob_or_pred if prob_or_pred.ndim == 1 else prob_or_pred.argmax(1)
    idx = np.arange(len(ytrue)) if idx is None else idx
    y, p, b = ytrue[idx], pred[idx], BLOK_TR[idx]
    out = 0.0
    for m, w in ((b, W_BLOK), (~b, 1 - W_BLOK)):
        if m.sum() >= NC: out += w * macro_f1(y[m], p[m])
    return out

def popf1(p): return popf1_idx(p)

_rng = np.random.default_rng(0); _pp = _rng.integers(0, NC, len(ytrue))
assert abs(macro_f1(ytrue, _pp) - f1_score(ytrue, _pp, average="macro")) < 1e-9
print("macro_f1 cepat identik dengan sklearn.")

### 1.6 Cache citra

Setiap citra didekode sekali, diubah ke abu-abu, diperkecil sampai sisi
terpanjangnya 1.280 piksel, lalu disimpan sebagai PNG. Semua backbone membaca
dari cache yang sama. Formatnya identik dengan v13 sampai v17, jadi di Colab
cache lama di Drive langsung terpakai. Tidak ada Otsu atau binarisasi: dua uji
terkontrol di proyek ini mengukur keduanya merugikan (-0,0131 di DiT dan
-0,0091 di SigLIP2).

In [ ]:
CACHE = dict(enable=True)
def _cp(path): return os.path.join(CDIR, hashlib.md5(path.encode()).hexdigest() + ".png")
def load_img(path):
    if CACHE["enable"]:
        c = _cp(path)
        if os.path.exists(c):
            try: return Image.open(c).convert("L")
            except Exception: pass
    try:
        im = Image.open(path); im.draft("L", (CACHE_MAX, CACHE_MAX)); g = im.convert("L")
    except Exception:
        return Image.new("L", (64, 64), 255)
    if max(g.size) > CACHE_MAX: g.thumbnail((CACHE_MAX, CACHE_MAX), Image.BILINEAR)
    if CACHE["enable"]:
        try: g.save(_cp(path), "PNG", optimize=False)
        except Exception: pass
    return g

_t0, _n = time.time(), 0
for _p in tr_df.path.tolist() + te_df.path.tolist():
    if not os.path.exists(_cp(_p)): load_img(_p); _n += 1
print(f"{_n} citra baru didekode dalam {time.time()-_t0:.0f} detik {clk()}")

---
# Bagian 2. Daftar kandidat

Sebelum GPU dipakai, setiap nama diperiksa keberadaannya di registri timm.
Nama yang salah tidak diganti diam-diam dengan model lain. Pelajaran ini
datang dari v10, yang tanpa sadar jatuh ke resnet34 ketika nama backbone
salah ketik, sehingga run yang dikira DINOv2 sebenarnya ResNet.

In [ ]:
_ada = set(timm.list_pretrained())
tab_kand = pd.DataFrame([dict(tag=c["tag"], keluarga=c["fam"], resolusi=c["h"], prio=c["prio"],
                              sumber=c["src"],
                              di_timm=("ya" if c["src"] != "timm" or c["name"] in _ada
                                       else "TIDAK"),
                              nama=c["name"]) for c in CANDIDATES])
display(tab_kand.sort_values(["prio", "keluarga", "tag"]).reset_index(drop=True))
_hilang = tab_kand[tab_kand.di_timm == "TIDAK"]
if len(_hilang):
    print(f"\n{len(_hilang)} nama tidak dikenal timm {timm.__version__}: {list(_hilang.tag)}")
    print("Kandidat itu akan dicatat GAGAL, bukan diganti. Biasanya obatnya timm lebih baru.")
print(f"\nkeluarga: {tab_kand.keluarga.value_counts().to_dict()}")

---
# Bagian 3. Ekstraksi fitur beku

### 3.1 Dua pooling dari satu lintasan maju

Setiap backbone dibekukan dan dijalankan sekali atas seluruh citra train dan
test. Dari satu lintasan itu diambil dua vektor fitur. Yang pertama adalah
pooling bawaan backbone, persis seperti yang dipakai v1 dan seperti yang
dilihat kepala klasifikasi saat fine-tune. Yang kedua adalah pooling
alternatif: rata-rata token patch untuk transformer (token CLS dan register
dibuang), atau GeM dengan p=3 untuk jaringan konvolusi. Untuk backbone yang
pooling bawaannya sudah rata-rata token (DINOv3, misalnya), alternatifnya
adalah token CLS.

Alasannya spesifik untuk data ini. Aksara dibedakan oleh bentuk goresan kecil
yang tersebar di seluruh citra, dan token CLS atau attention pooling bisa saja
meringkas citra dengan cara yang terlalu global. Kalau pooling rata-rata
memberi skor jauh lebih tinggi untuk suatu backbone, itu petunjuk langsung
untuk kepala fine-tune-nya. Biayanya nol, karena lintasan majunya sama.

Kesetaraan pooling bawaan dengan keluaran model diperiksa di batch pertama.
Kalau tidak sama (arsitektur yang `forward_head`-nya tidak standar), notebook
kembali ke `model(x)` untuk fitur bawaan dan mencatatnya.

Selain fitur, sel ini mengukur biaya: jumlah parameter, citra per detik, dan
VRAM puncak. Ketiganya dipakai di Bagian 6 untuk kurva biaya lawan skor.

In [ ]:
MEAN_DEF, STD_DEF = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
def letterbox(g, H, W):
    w, h = g.size
    s = min(H / max(h, 1), W / max(w, 1))
    nw, nh = max(1, min(W, int(round(w*s)))), max(1, min(H, int(round(h*s))))
    g = g.resize((nw, nh), Image.BILINEAR)
    c = Image.new("L", (W, H), 255); c.paste(g, ((W-nw)//2, (H-nh)//2)); return c
TO_T = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

class EvalDS(Dataset):
    def __init__(s, paths, r):
        s.p = list(paths); s.r = r
        s.norm = T.Normalize(r.get("mean", MEAN_DEF), r.get("std", STD_DEF))
    def __len__(s): return len(s.p)
    def __getitem__(s, i):
        x = TO_T(letterbox(load_img(s.p[i]), s.r["h"], s.r["w"]))
        if x.shape[0] == 1: x = x.repeat(3, 1, 1)
        return s.norm(x), 0

def hf_pool(out):
    po = getattr(out, "pooler_output", None)
    if po is not None and getattr(po, "ndim", 0) == 2: return po
    h = out.last_hidden_state
    if h.ndim == 4: return h.mean((2, 3))
    if h.shape[1] > 1: return h[:, 1:].mean(1)
    return h.mean(1)

class HFBody(nn.Module):
    def __init__(s, m):
        super().__init__(); s.m = m
        try: s.ipe = "interpolate_pos_encoding" in inspect.signature(m.forward).parameters
        except Exception: s.ipe = False
    def forward(s, x):
        return hf_pool(s.m(pixel_values=x, interpolate_pos_encoding=True)) if s.ipe \
               else hf_pool(s.m(pixel_values=x))

def load_body(c):
    """(body, dim, mean, std) atau None. Kandidat yang gagal DILEWATI, tidak diganti."""
    try:
        if c["src"] == "timm":
            def _mk(pre):
                try:
                    return timm.create_model(c["name"], pretrained=pre, num_classes=0,
                                             img_size=(c["h"], c["w"]))
                except TypeError:
                    return timm.create_model(c["name"], pretrained=pre, num_classes=0)
            try: m = _mk(True)
            except Exception as e:
                if not SMOKE: raise
                print(f"      (SMOKE) unduhan gagal {type(e).__name__} -> bobot ACAK")
                m = _mk(False)
            mean, std = MEAN_DEF, STD_DEF
            try:
                from timm.data import resolve_data_config
                dc = resolve_data_config({}, model=m)
                mean, std = list(dc.get("mean", mean)), list(dc.get("std", std))
            except Exception: pass
            return m, int(m.num_features), mean, std
        if SMOKE: raise RuntimeError("SMOKE: kandidat Hugging Face dilewati")
        from transformers import AutoModel, AutoConfig
        kw = {}
        cf = AutoConfig.from_pretrained(c["name"])
        if getattr(cf, "model_type", "") == "vit_mae": kw["mask_ratio"] = 0.0
        m = AutoModel.from_pretrained(c["name"], **kw)
        mean, std = MEAN_DEF, STD_DEF
        try:
            from transformers import AutoImageProcessor
            ip = AutoImageProcessor.from_pretrained(c["name"])
            if getattr(ip, "image_mean", None): mean = list(ip.image_mean)
            if getattr(ip, "image_std", None):  std = list(ip.image_std)
        except Exception: pass
        dim = int(getattr(m.config, "hidden_size", 0) or
                  (m.config.hidden_sizes[-1] if getattr(m.config, "hidden_sizes", None) else 0))
        if dim <= 0: raise RuntimeError("dimensi fitur tidak terbaca dari config")
        return HFBody(m), dim, mean, std
    except Exception as e:
        print(f"      !! DILEWATI: {c['src']}:{c['name']} -> {type(e).__name__}: {str(e)[:300]}")
        return None

def _is_oom(e):
    return isinstance(e, getattr(torch.cuda, "OutOfMemoryError", ())) \
           or ("out of memory" in str(e).lower())

def pool_alt(m, ff):
    """Pooling alternatif: rata-rata token patch (transformer) atau GeM p=3 (konvolusi)."""
    if not torch.is_tensor(ff): return None   # contoh: CrossViT dan CoaT mengembalikan list
    nf = int(m.num_features)
    if ff.ndim == 3:
        npt = int(getattr(m, "num_prefix_tokens", 0) or 0)
        tok = ff[:, npt:] if ff.shape[1] > npt else ff
        return tok.float().mean(1)
    if ff.ndim == 4:
        if ff.shape[1] == nf: t = ff
        elif ff.shape[-1] == nf: t = ff.permute(0, 3, 1, 2)
        else: return None
        return t.float().clamp(min=1e-6).pow(3).mean((2, 3)).pow(1.0 / 3)
    return None

# --- pandangan siap pakai per resolusi ----------------------------------------
# Dengan ratusan backbone, biaya terbesar ekstraksi bukan GPU tetapi CPU:
# membaca PNG, letterbox, dan normalisasi diulang untuk setiap backbone. Karena
# letterbox deterministik, hasilnya sama untuk semua backbone yang resolusinya
# sama. Jadi citra di-letterbox SEKALI per resolusi, disimpan di RAM sebagai
# uint8 abu-abu, lalu normalisasi dikerjakan di GPU. Isinya identik piksel demi
# piksel dengan EvalDS; yang berubah cuma di mana pekerjaan itu dilakukan.
from concurrent.futures import ThreadPoolExecutor
VIEWS = OrderedDict()
def _lb_arr(p, H, W): return np.asarray(letterbox(load_img(p), H, W), dtype=np.uint8)
def view(H, W):
    key = (H, W)
    if key in VIEWS:
        VIEWS.move_to_end(key); return VIEWS[key]
    while len(VIEWS) >= 2: VIEWS.popitem(last=False)
    t0 = time.time(); paths = tr_df.path.tolist() + te_df.path.tolist()
    with ThreadPoolExecutor(max(2, (os.cpu_count() or 2))) as ex:
        arr = np.stack(list(ex.map(lambda p: _lb_arr(p, H, W), paths, chunksize=32)))
    VIEWS[key] = arr
    print(f"      pandangan {H}x{W} disiapkan: {arr.nbytes/2**30:.2f} GB, {time.time()-t0:.0f}s")
    return arr

def buang_bobot(c):
    """Hapus bobot yang baru diunduh. 330 backbone x rata-rata 300 MB tidak muat di
    disk Kaggle, jadi setiap bobot dibuang begitu fiturnya tersimpan."""
    import shutil
    stem = c["name"].split(".")[0]
    for root in (os.path.expanduser("~/.cache/huggingface/hub"),
                 os.path.expanduser("~/.cache/torch/hub/checkpoints")):
        if not os.path.isdir(root): continue
        for d in os.listdir(root):
            if stem in d or c["name"].replace("/", "--") in d:
                try:
                    p = os.path.join(root, d)
                    shutil.rmtree(p) if os.path.isdir(p) else os.remove(p)
                except Exception: pass

@torch.no_grad()
def extract(body, is_timm, rows, r, bs, state):
    """rows: indeks ke pandangan gabungan train+test. Kembalikan (bawaan, alt)."""
    V = view(r["h"], r["w"])
    mean = torch.tensor(r["mean"], device=DEV).view(1, 3, 1, 1)
    std = torch.tensor(r["std"], device=DEV).view(1, 3, 1, 1)
    A, B, Ccls = [], [], []
    for i in range(0, len(rows), bs):
        xb = torch.from_numpy(V[rows[i:i+bs]]).to(DEV, non_blocking=True)
        x = ((xb.float() / 255.0).unsqueeze(1).repeat(1, 3, 1, 1) - mean) / std
        with torch.autocast(DEV, torch.float16, enabled=(DEV == "cuda")):
            if not is_timm or state.get("dua_lintasan"):
                A.append(body(x).float().cpu().numpy()); continue
            ff = body.forward_features(x)
            d = body.forward_head(ff, pre_logits=True).float()
            if "cek" not in state:
                ref = body(x).float()
                state["cek"] = bool(torch.allclose(ref, d, atol=2e-2, rtol=2e-2))
                if not state["cek"]:
                    print("      forward_head tidak setara model(x): fitur bawaan diambil "
                          "dari model(x), pooling alternatif dimatikan")
                    state["dua_lintasan"] = True
                    A.append(ref.cpu().numpy()); continue
            A.append(d.cpu().numpy())
            a = pool_alt(body, ff)
            if a is not None: B.append(a.float().cpu().numpy())
            if torch.is_tensor(ff) and ff.ndim == 3 and int(getattr(body, "num_prefix_tokens", 0) or 0) > 0:
                Ccls.append(ff[:, 0].float().cpu().numpy())
    A = np.concatenate(A)
    B = np.concatenate(B) if (B and sum(len(b) for b in B) == len(A)) else None
    if B is not None and np.allclose(A, B, atol=1e-3):
        # Pooling bawaan backbone ini memang rata-rata token (contoh: DINOv3 dan
        # siglip2-gap), jadi alternatifnya diganti token CLS kalau ada.
        B = np.concatenate(Ccls) if Ccls else None
    return A, B

# --- penyimpanan fitur dan lanjutan lintas sesi -------------------------------
# Fitur disimpan float16 (setengah ukuran, selisihnya jauh di bawah derau probe).
# Berkas dari sesi sebelumnya dicari di FDIR dan juga di seluruh /kaggle/input,
# sehingga sesi kedua cukup melampirkan output sesi pertama sebagai input.
def _fname(c): return f"{c['tag']}_{c['h']}x{c['w']}.npz"
FSRC = {}
for _root in [FDIR] + (["/kaggle/input"] if IS_KAGGLE else []):
    if not os.path.isdir(_root): continue
    for dp, dn, fs in os.walk(_root, followlinks=True):
        for f in fs:
            if f.endswith(".npz") and os.path.basename(dp) == "_fitur":
                FSRC.setdefault(f, os.path.join(dp, f))
print(f"berkas fitur yang sudah ada (sesi ini atau sesi sebelumnya): {len(FSRC)}")

_LRU = OrderedDict()
def load_feat(tag):
    """Muat fitur satu kandidat dari disk (float32), dengan cache kecil di RAM.
    Ratusan kandidat tidak disimpan sekaligus di memori."""
    if tag in _LRU:
        _LRU.move_to_end(tag); return _LRU[tag]
    z = np.load(FEAT[tag]["path"], allow_pickle=True)
    d = dict(tr=z["tr"].astype(np.float32), te=z["te"].astype(np.float32))
    if "tr_alt" in z.files:
        d["tr_alt"], d["te_alt"] = z["tr_alt"].astype(np.float32), z["te_alt"].astype(np.float32)
    _LRU[tag] = d
    while len(_LRU) > 6: _LRU.popitem(last=False)
    return d

def feats_for(c):
    """Pastikan fitur satu kandidat ada di disk. Kembalikan dict meta atau None."""
    fn = _fname(c); fp = os.path.join(FDIR, fn)
    src = FSRC.get(fn)
    if src:
        try:
            z = np.load(src, allow_pickle=True)
            if len(z["tr"]) == len(tr_df) and len(z["te"]) == len(te_df):
                return dict(path=src, cached=True, meta=json.loads(str(z["meta"])),
                            alt="tr_alt" in z.files)
            print("      ukuran cache tidak cocok, diekstrak ulang")
        except Exception as e:
            print(f"      cache rusak ({type(e).__name__}), diekstrak ulang")
    t_load = time.time()
    got = load_body(c)
    if got is None: return None
    body, dim, mean, std = got
    t_load = time.time() - t_load
    n_par = sum(p.numel() for p in body.parameters()) / 1e6
    r = dict(h=c["h"], w=c["w"], mean=mean, std=std)
    body = body.to(DEV).eval()
    if DEV == "cuda": torch.cuda.reset_peak_memory_stats()
    bs = CFG["batch"] * 2 if c["mult"] < 1.5 and c["h"] <= 256 else \
         (CFG["batch"] if c["mult"] < 3 else CFG["batch"] // 2)
    n_tr = len(tr_df); rows = np.arange(n_tr + len(te_df))
    view(c["h"], c["w"])   # dibangun di luar pengukuran kecepatan
    state = {}
    while True:
        try:
            t0 = time.time()
            a, b = extract(body, c["src"] == "timm", rows, r, bs, state)
            t_fwd = time.time() - t0
            break
        except Exception as e:
            if not _is_oom(e) or bs <= 2:
                del body
                if DEV == "cuda": torch.cuda.empty_cache()
                buang_bobot(c)
                print(f"      !! ekstraksi gagal: {type(e).__name__}: {str(e)[:300]}")
                return None
            if DEV == "cuda": torch.cuda.empty_cache()
            bs //= 2; state = {}; print(f"      OOM, batch diturunkan ke {bs}")
    vram = torch.cuda.max_memory_allocated() / 2**30 if DEV == "cuda" else float("nan")
    del body
    if DEV == "cuda": torch.cuda.empty_cache()
    buang_bobot(c)
    if not np.isfinite(a).all():
        print("      !! fitur mengandung NaN atau inf (biasanya overflow float16), kandidat dibuang")
        return None
    meta = dict(params_m=round(n_par, 2), img_per_s=round(len(rows) / max(t_fwd, 1e-6), 2),
                vram_gb=round(vram, 3), sec_fwd=round(t_fwd, 1), sec_load=round(t_load, 1),
                dim=int(a.shape[1]), batch=bs, mean=mean, std=std)
    arrs = dict(tr=a[:n_tr].astype(np.float16), te=a[n_tr:].astype(np.float16))
    if b is not None and np.isfinite(b).all():
        arrs["tr_alt"], arrs["te_alt"] = b[:n_tr].astype(np.float16), b[n_tr:].astype(np.float16)
    try:
        np.savez(fp, meta=json.dumps(meta), **arrs)
    except Exception as e:
        print(f"      !! gagal menyimpan fitur: {type(e).__name__}"); return None
    FSRC[fn] = fp
    return dict(path=fp, cached=False, meta=meta, alt="tr_alt" in arrs)

### 3.2 Mesin probe: regresi logistik di GPU

v1 memakai `LogisticRegression` dari sklearn di CPU, dan itulah alasan hanya
lima kandidat yang sempat ditala. Di v2 regresi logistiknya ditulis ulang di
PyTorch dan dioptimasi dengan L-BFGS, dengan fungsi objektif yang sama persis
dengan sklearn: entropi silang berbobot kelas seimbang ditambah penalti L2
sebesar 1/(2C) pada bobot (bias tidak dipenalti). Dengan begitu satu setelan
kali 5 fold selesai dalam sepersekian detik, dan menala 333 kandidat menjadi
urusan satu jam, bukan satu hari.

Karena mesinnya baru, kesetaraannya dengan sklearn dibuktikan di awal
Bagian 4 pada salah satu kandidat. Kalau selisihnya besar, semua angka di
bawahnya tidak boleh dipercaya, dan sel itu akan mengatakannya.

Tiga pilihan normalisasi diuji: `l2` (setiap vektor dinormalkan panjangnya),
`std` (setiap dimensi distandarkan dengan rerata dan simpangan fold latih),
dan `raw` (apa adanya). Normalisasi `std` selalu dihitung dari fold latih
saja supaya tidak ada informasi fold validasi yang bocor.

In [ ]:
PDEV = DEV
PDT = torch.float64

def _norm_pair(Xa, Xb, norm):
    if norm == "l2":
        return (Xa / Xa.norm(dim=1, keepdim=True).clamp_min(1e-9),
                Xb / Xb.norm(dim=1, keepdim=True).clamp_min(1e-9))
    if norm == "std":
        mu = Xa.mean(0, keepdim=True); sd = Xa.std(0, keepdim=True).clamp_min(1e-6)
        return (Xa - mu) / sd, (Xb - mu) / sd
    return Xa, Xb

def fit_lr(X, y, C, nc, W0=None, iters=300):
    """Setara LogisticRegression(C, class_weight='balanced') multinomial sklearn."""
    n, d = X.shape
    Xb = torch.cat([X, torch.ones(n, 1, device=X.device, dtype=X.dtype)], 1)
    cnt_ = torch.bincount(y, minlength=nc).to(X.dtype)
    sw = (n / (nc * cnt_.clamp_min(1)))[y] / n
    W = (W0.clone() if W0 is not None else torch.zeros(d + 1, nc, device=X.device, dtype=X.dtype))
    W.requires_grad_(True)
    opt = torch.optim.LBFGS([W], lr=1.0, max_iter=iters, history_size=20,
                            tolerance_grad=1e-7, tolerance_change=1e-10,
                            line_search_fn="strong_wolfe")
    lam = 0.5 / (C * n)
    def closure():
        opt.zero_grad()
        loss = (F.cross_entropy(Xb @ W, y, reduction="none") * sw).sum() + lam * (W[:-1] ** 2).sum()
        loss.backward(); return loss
    opt.step(closure)
    return W.detach()

def pred_lr(W, X):
    Xb = torch.cat([X, torch.ones(len(X), 1, device=X.device, dtype=X.dtype)], 1)
    return torch.softmax(Xb @ W, 1)

def cv_lr(Xnp, folds, norms=("l2",), Cs=(1.0,), y=None, nc=None):
    """OOF untuk semua kombinasi (norm, C). Awal hangat dari C sebelumnya."""
    y = ytrue if y is None else y; nc = nc or NC
    X = torch.as_tensor(Xnp, device=PDEV, dtype=PDT); yt = torch.as_tensor(y, device=PDEV)
    out = {(nm, C): np.zeros((len(y), nc)) for nm in norms for C in Cs}
    for itr, iva in folds:
        itr_t = torch.as_tensor(itr, device=PDEV); iva_t = torch.as_tensor(iva, device=PDEV)
        for nm in norms:
            Xa, Xv = _norm_pair(X[itr_t], X[iva_t], nm)
            W = None
            for C in sorted(Cs):
                W = fit_lr(Xa, yt[itr_t], C, nc, W0=W)
                out[(nm, C)][iva] = pred_lr(W, Xv).cpu().numpy()
    return out

def fit_predict_lr(Xtr_np, Xte_np, norm, C):
    X = torch.as_tensor(Xtr_np, device=PDEV, dtype=PDT)
    Z = torch.as_tensor(Xte_np, device=PDEV, dtype=PDT)
    Xa, Zv = _norm_pair(X, Z, norm)
    W = fit_lr(Xa, torch.as_tensor(ytrue, device=PDEV), C, NC)
    return pred_lr(W, Zv).cpu().numpy()

### 3.3 Sapuan

Gubernur waktu memperkirakan biaya setiap kandidat sebelum memulainya dan
melewatinya kalau sisa waktu tidak cukup, dengan tetap menyisakan jatah untuk
penalaan (Bagian 4) dan konfirmasi fine-tune (Bagian 8). Perkiraan awalnya
kasar, tetapi setelah kandidat pertama selesai kecepatan sungguhan GPU ini
terukur dan perkiraan berikutnya memakai angka itu.

Tiap baris yang dicetak memuat `popf1` probe cepat (regresi logistik dengan
setelan tetap, C=16 tanpa normalisasi, yaitu setelan juara v1). Angka ini
hanya untuk memantau jalannya sapuan. Peringkat yang sebenarnya menunggu
penalaan penuh di Bagian 4.

In [ ]:
FEAT, RES = OrderedDict(), OrderedDict()
_rate = None   # detik per (citra x satuan resolusi x mult), diukur dari kandidat pertama

def est_sec(c):
    n = len(tr_df) + len(te_df)
    mp = (c["h"] * c["w"]) / (224.0 * 224.0)
    return n * mp * c["mult"] * (_rate if _rate is not None else 0.0025) + 25.0

def jatah_ft_h():
    if not CONFIRM["enable"]: return 0.0
    return 0.30 * CONFIRM["k"] * CONFIRM["epochs"] / 6.0

def quick_probe(X):
    """Probe cepat untuk pemantauan saja (setelan juara v1: C=16 tanpa normalisasi)."""
    return popf1(cv_lr(X, FOLDS[SEED], norms=("raw",), Cs=(16.0,))[("raw", 16.0)])

URUT = sorted(CANDIDATES, key=lambda d: (d["prio"], d["h"], d["w"], d["tag"]))
_sisa_awal = sum(est_sec(c) for c in URUT if _fname(c) not in FSRC) / 3600
print(f"STEP 3 - SAPUAN {len(CANDIDATES)} KANDIDAT {clk()}")
print(f"  {sum(_fname(c) in FSRC for c in URUT)} sudah punya fitur dari sesi sebelumnya; "
      f"perkiraan kasar sisa ekstraksi {_sisa_awal:.1f} jam\n")
for k, c in enumerate(URUT):
    ada = _fname(c) in FSRC
    need = 0.0 if ada else est_sec(c) / 3600.0
    sisa = rem_h() - BUDGET["reserve_h"] - BUDGET["probe_h"] - jatah_ft_h()
    if need * BUDGET["safety"] > sisa:
        RES[c["tag"]] = dict(tag=c["tag"], status="belum (anggaran sesi habis)")
        continue
    t0 = time.time()
    d = feats_for(c)
    if d is None:
        RES[c["tag"]] = dict(tag=c["tag"], status="GAGAL dimuat")
        print(f"  [{k+1:3d}/{len(URUT)}] {c['tag']:34s} GAGAL"); continue
    if not d["cached"]:
        mp = (c["h"] * c["w"]) / (224.0 * 224.0)
        r_new = d["meta"]["sec_fwd"] / ((len(tr_df) + len(te_df)) * mp * max(c["mult"], 0.05))
        _rate = r_new if _rate is None else 0.8 * _rate + 0.2 * r_new
    FEAT[c["tag"]] = d
    try: q = quick_probe(load_feat(c["tag"])["tr"])
    except Exception as e: q = float("nan"); print(f"      probe cepat gagal: {type(e).__name__}")
    RES[c["tag"]] = dict(tag=c["tag"], status="ok" + (" (cache)" if d["cached"] else ""),
                         popf1_cepat=q)
    m = d["meta"]
    print(f"  [{k+1:3d}/{len(URUT)}] {c['tag']:34s} @{c['h']:<3d} popf1 cepat {q:.4f} | "
          f"{m['params_m']:5.0f} jt | {m['img_per_s']:5.0f} citra/s | "
          f"{'cache' if d['cached'] else f'{time.time()-t0:4.0f}s'} {clk()}", flush=True)

OKT = [t for t in RES if RES[t]["status"].startswith("ok")]
BELUM = [t for t in RES if RES[t]["status"].startswith("belum")]
_gagal = [t for t in RES if RES[t]["status"].startswith("GAGAL")]
print(f"\n{len(OKT)} kandidat punya fitur, {len(_gagal)} gagal, {len(BELUM)} belum sempat. {clk()}")
if _gagal: print(f"gagal: {_gagal}")
if BELUM:
    print(f"\n!! {len(BELUM)} kandidat BELUM diekstrak karena anggaran sesi ini habis.")
    print("   Analisis di bawah tetap jalan untuk kandidat yang sudah ada. Untuk")
    print("   menyelesaikan sisanya, lihat petunjuk sesi lanjutan di Bagian 1.3.")
if not OKT and BELUM:
    raise RuntimeError("anggaran sesi habis sebelum satu kandidat pun diekstrak; naikkan INFEST_BUDGET_H")
if not OKT: raise RuntimeError("tidak ada kandidat yang berhasil; periksa Internet dan pesan di atas")
CAND = {c["tag"]: c for c in CANDIDATES}
VIEWS.clear()

---
# Bagian 4. Penalaan yang adil

### 4.1 Bukti mesin probe setara sklearn

Sebelum 30 setelan dijalankan untuk setiap kandidat, hasil mesin GPU
dibandingkan dengan sklearn di dua setelan pada satu kandidat. Selisih
popf1 di bawah 0,004 dan kesepakatan prediksi di atas 98% berarti mesin ini
boleh dipakai sebagai pengganti.

In [ ]:
# --- bukti kesetaraan dengan sklearn ----------------------------------------
_t = "siglip2-384" if "siglip2-384" in FEAT else OKT[0]
_X = load_feat(_t)["tr"]
_t0 = time.time(); _o = cv_lr(_X, FOLDS[SEED], norms=("l2", "raw"), Cs=(1.0, 16.0))
_dt_torch = time.time() - _t0
print(f"cek kesetaraan pada {_t}:")
print(f"  {'setelan':14s}{'torch':>9s}{'sklearn':>9s}{'selisih':>9s}{'setuju':>9s}")
_maxdiff = 0.0
for nm, C in (("l2", 1.0), ("raw", 16.0)):
    Xs = _X / np.maximum(np.linalg.norm(_X, axis=1, keepdims=True), 1e-9) if nm == "l2" else _X
    ref = np.zeros((len(Xs), NC))
    for itr, iva in FOLDS[SEED]:
        lr = LogisticRegression(C=C, max_iter=1000, class_weight="balanced")
        lr.fit(Xs[itr], ytrue[itr]); ref[np.ix_(iva, lr.classes_)] = lr.predict_proba(Xs[iva])
    a, b = popf1(_o[(nm, C)]), popf1(ref)
    agree = (_o[(nm, C)].argmax(1) == ref.argmax(1)).mean()
    _maxdiff = max(_maxdiff, abs(a - b))
    print(f"  {nm+' C='+str(C):14s}{a:9.4f}{b:9.4f}{a-b:+9.4f}{agree*100:8.1f}%")
print(f"  waktu torch untuk 4 setelan x 5 fold: {_dt_torch:.1f} detik")
if _maxdiff > 0.004 and not SMOKE:
    print("  !! selisih lebih dari 0,004. Mesin probe GPU TIDAK setara sklearn di sini,")
    print("     jadi peringkat di bawah harus dibaca dengan curiga.")
else:
    print("  -> setara. Peringkat di bawah boleh dibaca seperti peringkat sklearn.")

### 4.2 Penalaan dua tingkat untuk SEMUA kandidat

Setiap kandidat, tanpa kecuali, melewati kisi tingkat pertama: dua pooling,
dua normalisasi (`l2` dan `raw`), tiga nilai C (4, 16, 64), jadi 12 setelan
kali 5 fold. Ini memperbaiki kelemahan terbesar v1, yang hanya menala lima
kandidat teratas padahal peringkat sebelum ditala terbukti menyesatkan.

Empat puluh kandidat teratas setelah tingkat pertama lalu masuk kisi penuh:
dua pooling, tiga normalisasi (ditambah `std`), lima nilai C dari 1 sampai
256, jadi 30 setelan. Kisi penuh untuk semua 330 kandidat akan memakan
beberapa jam tanpa mengubah siapa yang ada di papan atas, karena kandidat di
bawah posisi 40 tertinggal jauh lebih besar dari untung penalaan mana pun
(di v1 untung terbesar dari menala sekitar 0,07, dan itu terjadi di papan
atas juga, jadi urutan relatifnya terjaga).

Untuk setiap kandidat, dalam satu kali memuat fiturnya, sel ini juga
mengerjakan hal-hal yang nanti dibutuhkan bagian lain: pengukuran ulang di
dua pembagian fold lain (Bagian 4.3), AUC jurang domain (Bagian 5.3), kNN
dan pusat kelas (Bagian 6), serta prediksi test (Bagian 10). Hasilnya
disimpan per kandidat di `automl2/_tala/`, sehingga sesi lanjutan tidak
menala ulang kandidat yang sudah selesai.

Memilih yang terbaik dari banyak setelan selalu sedikit menggelembungkan
angka. Karena itu setelan terbaik dibekukan lalu diukur ulang di dua
pembagian fold baru (seed 7 dan 2026) yang tidak ikut memilih. Kolom
`popf1_ulang` adalah rata-rata kedua pengukuran itu, `bias_pilih` adalah
selisihnya dengan skor tertala, dan `sd_fold` adalah simpangan antar ketiga
pembagian, yaitu ukuran derau yang dihitung dari data.

In [ ]:
TDIR = os.path.join(OUT, "_tala"); os.makedirs(TDIR, exist_ok=True)
TSRC = {}
for _root in [TDIR] + (["/kaggle/input"] if IS_KAGGLE else []):
    if not os.path.isdir(_root): continue
    for dp, dn, fs in os.walk(_root, followlinks=True):
        if os.path.basename(dp) == "_tala":
            for f in fs:
                if f.endswith(".npz"): TSRC.setdefault(f, os.path.join(dp, f))

def feat_of(tag, pool, split="tr"):
    d = load_feat(tag)
    return d.get(split + "_alt") if pool == "alt" else d[split]

nd = len(tr_df) + len(te_df)
ydom = np.r_[np.zeros(len(tr_df), int), np.ones(len(te_df), int)]
FOLD_DOM = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(np.zeros(nd), ydom))

def _gsig(g): return "p" + "".join(x[0] for x in g["pool"]) + "n" + "".join(x[0] for x in g["norm"]) \
                     + "c" + "-".join(f"{c:g}" for c in g["C"])

def analisis_kandidat(t, grid):
    fn = f"{t}__{_gsig(grid)}.npz"
    if fn in TSRC:
        try:
            z = np.load(TSRC[fn], allow_pickle=True)
            return json.loads(str(z["info"])), {k: z[k] for k in z.files if k != "info"}
        except Exception: pass
    rows, best, oofs = [], None, {}
    for pool in grid["pool"]:
        X = feat_of(t, pool)
        if X is None: continue
        for (nm, C), oof in cv_lr(X, FOLDS[SEED], norms=grid["norm"], Cs=grid["C"]).items():
            s = popf1(oof)
            rows.append(dict(tag=t, pool=pool, norm=nm, C=C, popf1=s,
                             f1_macro=macro_f1(ytrue, oof.argmax(1))))
            if best is None or s > best[3]: best = (pool, nm, C, s); oofs["oof42"] = oof
    pool, nm, C, _ = best
    X = feat_of(t, pool)
    for s in SEEDS_ULANG:
        oofs[f"oof{s}"] = cv_lr(X, FOLDS[s], norms=(nm,), Cs=(C,))[(nm, C)]
    Xd = np.vstack([X, feat_of(t, pool, "te")])
    o = cv_lr(Xd, FOLD_DOM, norms=(nm,), Cs=(1.0,), y=ydom, nc=2)[(nm, 1.0)]
    dom = float(roc_auc_score(ydom, o[:, 1]))
    r = cv_knn_ncm(X, FOLDS[SEED])
    kb = max(KNN_K, key=lambda k: popf1(r[("knn", k)]))
    oofs["test"] = fit_predict_lr(X, feat_of(t, pool, "te"), nm, C)
    info = dict(best=list(best), rows=rows, dom=dom, knn=popf1(r[("knn", kb)]), knn_k=kb,
                ncm=popf1(r[("ncm", 0)]))
    try:
        np.savez_compressed(os.path.join(TDIR, fn), info=json.dumps(info),
                            **{k: v.astype(np.float32) for k, v in oofs.items()})
        TSRC[fn] = os.path.join(TDIR, fn)
    except Exception as e: print(f"      ! gagal menyimpan hasil tala {t}: {type(e).__name__}")
    return info, oofs

def _l2t(X): return X / X.norm(dim=1, keepdim=True).clamp_min(1e-9)
def cv_knn_ncm(Xnp, folds):
    X = _l2t(torch.as_tensor(Xnp, device=PDEV, dtype=torch.float32))
    yt = torch.as_tensor(ytrue, device=PDEV)
    out = {("knn", k): np.zeros((len(ytrue), NC)) for k in KNN_K}
    out[("ncm", 0)] = np.zeros((len(ytrue), NC))
    for itr, iva in folds:
        A, B = X[itr], X[iva]; ya = yt[itr]
        S = B @ A.T
        for k in KNN_K:
            v, ix = S.topk(min(k, len(itr)), 1)
            w = torch.softmax(v / 0.05, 1)
            pr = torch.zeros(len(iva), NC, device=PDEV).scatter_add_(1, ya[ix], w)
            out[("knn", k)][iva] = pr.cpu().numpy()
        M = torch.stack([_l2t(A[ya == c].mean(0, keepdim=True))[0] if (ya == c).any()
                         else torch.zeros(A.shape[1], device=PDEV) for c in range(NC)])
        out[("ncm", 0)][iva] = torch.softmax((B @ M.T) / 0.05, 1).cpu().numpy()
    return out

INFO, OOF, OOF_S, TEST_P = {}, {}, {}, {}
def _pakai(t, info, oofs):
    INFO[t] = info; OOF[t] = oofs["oof42"]; TEST_P[t] = oofs["test"]
    OOF_S[t] = {SEED: oofs["oof42"], **{s: oofs[f"oof{s}"] for s in SEEDS_ULANG}}

print(f"STEP 4a - PENALAAN TINGKAT 1: {len(OKT)} kandidat x "
      f"{len(GRID_CEPAT['pool'])*len(GRID_CEPAT['norm'])*len(GRID_CEPAT['C'])} setelan {clk()}")
_t0 = time.time()
for i, t in enumerate(OKT):
    info, oofs = analisis_kandidat(t, GRID_CEPAT); _pakai(t, info, oofs)
    if (i + 1) % 25 == 0 or i + 1 == len(OKT):
        print(f"  {i+1}/{len(OKT)} selesai, {(time.time()-_t0)/60:.1f} menit {clk()}", flush=True)

_urut1 = sorted(OKT, key=lambda t: -INFO[t]["best"][3])
PENUH = _urut1[:TOP_PENUH]
print(f"\nSTEP 4b - PENALAAN TINGKAT 2 (kisi penuh {len(GRID['pool'])*len(GRID['norm'])*len(GRID['C'])} "
      f"setelan) untuk {len(PENUH)} teratas {clk()}")
for t in PENUH:
    info, oofs = analisis_kandidat(t, GRID)
    if info["best"][3] >= INFO[t]["best"][3]:
        naik = info["best"][3] - INFO[t]["best"][3]
        info["rows"] = INFO[t]["rows"] + info["rows"]; _pakai(t, info, oofs)
    else:
        naik = 0.0
    b = INFO[t]["best"]
    _tepi = "  (C di tepi kisi)" if b[2] in (min(GRID["C"]), max(GRID["C"])) else ""
    print(f"  {t:34s} pool={b[0]:6s} norm={b[1]:3s} C={b[2]:<5g} popf1={b[3]:.4f} "
          f"(tingkat 2 {naik:+.4f}){_tepi}", flush=True)

BEST = {t: tuple(INFO[t]["best"]) for t in OKT}
KISI = pd.DataFrame([r for t in OKT for r in INFO[t]["rows"]]).drop_duplicates(
    ["tag", "pool", "norm", "C"], keep="last")
KISI.to_csv(os.path.join(OUT, "automl2_kisi.csv"), index=False)
rows = []
for t in OKT:
    sc = [popf1(OOF_S[t][s]) for s in (SEED,) + SEEDS_ULANG]
    rows.append(dict(tag=t, popf1_tuned=sc[0], popf1_ulang=float(np.mean(sc[1:])),
                     sd_fold=float(np.std(sc, ddof=1)), bias_pilih=sc[0] - float(np.mean(sc[1:]))))
ULANG = pd.DataFrame(rows).set_index("tag")
print(f"\npenalaan selesai dalam {(time.time()-_t0)/60:.1f} menit {clk()}")
print(f"rata-rata bias karena memilih setelan: {ULANG.bias_pilih.mean():+.4f}")
print(f"rata-rata simpangan antar pembagian fold: {ULANG.sd_fold.mean():.4f}")
display(ULANG.sort_values("popf1_ulang", ascending=False).round(4).head(25))

# jangkar terhadap v1: kalau angka siglip2-384 jauh dari v1, ada yang berubah
print("\nPembanding dengan v1 (setelan terbaik v1 adalah C=16 tanpa L2):")
for t, v in REF["v1_tuned"].items():
    if t in OKT:
        _raw16 = KISI[(KISI.tag == t) & (KISI.pool == "bawaan") & (KISI.norm == "raw")
                      & (KISI.C == 16.0)].popf1
        _r = float(_raw16.iloc[0]) if len(_raw16) else float("nan")
        print(f"  {t:14s} v1 {v:.4f} | setelan v1 di sini {_r:.4f} ({_r-v:+.4f}) | "
              f"tertala v2 {BEST[t][3]:.4f}")
print("Selisih kecil (di bawah 0,003) di kolom tengah berarti pipeline v2 mereproduksi v1.")

---
# Bagian 5. Perbandingan lengkap setiap kandidat

### 5.1 Metrik per kandidat

Satu angka tidak cukup untuk memilih backbone, karena dua backbone dengan
popf1 yang sama bisa gagal di tempat yang sangat berbeda. Tabel di bawah
memuat untuk setiap kandidat metrik-metrik berikut, semuanya dari OOF
setelan terbaiknya di seed 42.

Skor utamanya adalah `popf1` dan `f1_macro` (tanpa pembobotan populasi).
Skor per populasi, `f1_blok` dan `f1_strip`, penting karena blok adalah 49%
dari test tetapi populasi terlemah di semua versi sebelumnya. `acc`,
`bal_acc`, dan `top2` menunjukkan seberapa sering jawaban benar ada di tebakan
pertama atau kedua. `logloss` dan `ece` (galat kalibrasi, 15 kotak) mengukur
apakah probabilitasnya jujur, yang penting kalau probabilitas itu nanti
dirata-ratakan dalam ensemble. `jawi_ke_pegon` dan `pegon_ke_jawi` menghitung
pertukaran pasangan yang paling sulit, karena keduanya aksara Arab. Terakhir,
F1 untuk ketujuh kelas.

### 5.2 Ketidakpastian

`ci_lo` dan `ci_hi` adalah selang bootstrap 95% untuk popf1 (1.000 sampel
ulang baris OOF). `p_juara_lebih_baik` adalah proporsi sampel bootstrap di
mana juara mengalahkan kandidat itu, dihitung berpasangan pada baris yang
sama. Nilai di atas 0,95 berarti juara hampir pasti lebih baik. Nilai di
sekitar 0,5 sampai 0,8 berarti keduanya tidak bisa dibedakan dengan data
sebanyak ini. `p_mcnemar` adalah uji McNemar eksak pada benar atau salahnya
setiap baris.

In [ ]:
def ece_score(prob, y, bins=15):
    conf = prob.max(1); pred = prob.argmax(1); acc = (pred == y).astype(float)
    e = 0.0; edges = np.linspace(0, 1, bins + 1)
    for a, b in zip(edges[:-1], edges[1:]):
        m = (conf > a) & (conf <= b)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)

J, P = C2I.get("jawi"), C2I.get("pegon")
def metrik_lengkap(prob):
    pred = prob.argmax(1)
    pc = np.clip(prob[np.arange(len(ytrue)), ytrue], 1e-12, 1)
    rec = [((pred == k) & (ytrue == k)).sum() / max((ytrue == k).sum(), 1) for k in range(NC)]
    top2 = (np.argsort(-prob, 1)[:, :2] == ytrue[:, None]).any(1).mean()
    d = dict(popf1=popf1(prob), f1_macro=macro_f1(ytrue, pred),
             f1_blok=macro_f1(ytrue[BLOK_TR], pred[BLOK_TR]),
             f1_strip=macro_f1(ytrue[~BLOK_TR], pred[~BLOK_TR]),
             acc=float((pred == ytrue).mean()), bal_acc=float(np.mean(rec)), top2=float(top2),
             logloss=float(-np.log(pc).mean()), ece=ece_score(prob, ytrue),
             n_salah=int((pred != ytrue).sum()))
    if J is not None and P is not None:
        d["jawi_ke_pegon"] = int(((ytrue == J) & (pred == P)).sum())
        d["pegon_ke_jawi"] = int(((ytrue == P) & (pred == J)).sum())
    f = f1_score(ytrue, pred, average=None, labels=range(NC), zero_division=0)
    for k, cl in enumerate(CLASSES): d["f1_" + cl] = float(f[k])
    return d

rng = np.random.default_rng(BOOT["seed"])
BIDX = rng.integers(0, len(ytrue), size=(BOOT["n"], len(ytrue)))
def boot_scores(prob):
    pred = prob.argmax(1)
    return np.array([popf1_idx(pred, ix) for ix in BIDX])

_t0 = time.time()
BS = {t: boot_scores(OOF[t]) for t in OKT}
JUARA = max(OKT, key=lambda t: BEST[t][3])
rows = []
for t in OKT:
    c = CAND[t]; m = FEAT[t]["meta"]; pool, nm, C, s = BEST[t]
    d = dict(tag=t, keluarga=c["fam"], resolusi=c["h"], pool=pool, norm=nm, C=C)
    d.update(metrik_lengkap(OOF[t]))
    d["popf1_cepat"] = RES[t].get("popf1_cepat", np.nan)
    d.update(ULANG.loc[t].drop("popf1_tuned").to_dict())
    d["ci_lo"], d["ci_hi"] = np.percentile(BS[t], [2.5, 97.5])
    if t != JUARA:
        d["p_juara_lebih_baik"] = float((BS[JUARA] - BS[t] > 0).mean())
        cj = OOF[JUARA].argmax(1) == ytrue; ct = OOF[t].argmax(1) == ytrue
        b01, b10 = int((cj & ~ct).sum()), int((~cj & ct).sum())
        d["p_mcnemar"] = float(binomtest(b01, b01 + b10, 0.5).pvalue) if b01 + b10 else 1.0
    else:
        d["p_juara_lebih_baik"], d["p_mcnemar"] = np.nan, np.nan
    d.update(params_m=m["params_m"], img_per_s=m["img_per_s"], vram_gb=m["vram_gb"], dim=m["dim"])
    rows.append(d)
MASTER = pd.DataFrame(rows).set_index("tag").sort_values("popf1", ascending=False)
print(f"metrik dan bootstrap selesai dalam {time.time()-_t0:.0f} detik. Juara: {JUARA}")

### 5.3 Jurang domain di ruang fitur setiap backbone

v17 menemukan bahwa train dan test bisa dibedakan dari sifat berkasnya saja
dengan AUC 0,93. Pertanyaan yang relevan untuk memilih backbone adalah
apakah perbedaan itu ikut terbawa ke ruang fitur. Untuk setiap kandidat,
regresi logistik dilatih menebak apakah sebuah citra berasal dari train atau
test, hanya dari fiturnya. AUC 0,5 berarti fitur backbone itu tidak
membedakan keduanya, sedangkan AUC mendekati 1 berarti backbone itu sangat
peka terhadap perbedaan train dan test.

Angka ini menyaring, bukan memutuskan. Sebagian perbedaan train dan test bisa
jadi berupa sebaran kelas yang berbeda, yang memang sinyal. Tetapi di antara
dua backbone dengan popf1 setara, yang AUC domainnya lebih rendah lebih
mungkin mempertahankan skornya di papan.

In [ ]:
DOM = {t: INFO[t]["dom"] for t in OKT}
MASTER["auc_domain"] = pd.Series(DOM)
print("AUC domain (train lawan test di ruang fitur), makin rendah makin baik:")
print(MASTER.auc_domain.sort_values().round(3).to_string())

### 5.4 Tabel utama

Tabel lengkapnya ditulis ke `automl2_hasil.csv`. Yang ditampilkan di sini
adalah kolom-kolom yang paling sering dibutuhkan untuk memutuskan.

In [ ]:
MASTER.to_csv(os.path.join(OUT, "automl2_hasil.csv"))
_kol = ["keluarga", "resolusi", "pool", "norm", "C", "popf1", "ci_lo", "ci_hi", "popf1_ulang",
        "sd_fold", "f1_blok", "f1_strip", "jawi_ke_pegon", "pegon_ke_jawi", "logloss", "ece",
        "auc_domain", "p_juara_lebih_baik", "params_m", "img_per_s"]
display(MASTER[[k for k in _kol if k in MASTER.columns]].round(4))

PERKELAS = MASTER[["f1_" + c for c in CLASSES]].rename(columns=lambda s: s[3:])
PERKELAS.to_csv(os.path.join(OUT, "automl2_perkelas.csv"))
print("\nF1 per kelas (baris diurutkan menurut popf1):")
display(PERKELAS.round(4))

_setara = MASTER.index[(MASTER.p_juara_lebih_baik < 0.90).fillna(True)].tolist()
print(f"\nKandidat yang TIDAK bisa dibedakan dari juara {JUARA} (P juara lebih baik < 0,90):")
print(f"  {_setara}")

---
# Bagian 6. Apakah peringkatnya bergantung pada jenis kepala?

Regresi logistik adalah satu cara membaca ruang fitur, dan peringkat yang
dihasilkannya bisa saja kebetulan cocok dengan cara itu saja. Tiga kepala
lain dipakai sebagai pembanding, masing-masing menguji sifat ruang fitur yang
berbeda.

kNN kosinus (k dipilih dari 5, 10, 20) menguji apakah citra sekelas memang
berdekatan secara lokal, tanpa batas keputusan linier. Pusat kelas (nearest
class mean) adalah kepala paling sederhana: satu titik per kelas, dan skornya
tinggi hanya kalau setiap kelas membentuk satu gumpalan rapi. MLP kecil (satu
lapisan tersembunyi 512 unit) menguji apakah ada informasi yang baru bisa
dipakai dengan batas keputusan non-linier; ia dijalankan hanya untuk 15
kandidat teratas karena paling mahal.

Di akhir sel dihitung korelasi peringkat Kendall antara regresi logistik dan
tiap kepala. Tau di atas 0,7 berarti peringkatnya tahan terhadap pilihan
kepala, dan itu memperkuat keyakinan bahwa peringkat probe mencerminkan
kualitas backbone, bukan kecocokan dengan satu jenis pengklasifikasi.

In [ ]:
def cv_mlp(Xnp, folds, seed=SEED):
    X = torch.as_tensor(Xnp, device=PDEV, dtype=torch.float32)
    yt = torch.as_tensor(ytrue, device=PDEV)
    oof = np.zeros((len(ytrue), NC))
    for itr, iva in folds:
        torch.manual_seed(seed)
        itr_t = torch.as_tensor(itr, device=PDEV)
        Xa, Xv = _norm_pair(X[itr_t], X[torch.as_tensor(iva, device=PDEV)], "std")
        ya = yt[itr_t]
        cw = (len(ya) / (NC * torch.bincount(ya, minlength=NC).clamp_min(1).float()))
        net = nn.Sequential(nn.Linear(X.shape[1], MLP["hidden"]), nn.GELU(),
                            nn.Dropout(MLP["drop"]), nn.Linear(MLP["hidden"], NC)).to(PDEV)
        opt = torch.optim.AdamW(net.parameters(), lr=MLP["lr"], weight_decay=MLP["wd"])
        steps = MLP["epochs"] * math.ceil(len(ya) / 256)
        sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=MLP["lr"], total_steps=steps)
        crit = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.05)
        net.train()
        for ep in range(MLP["epochs"]):
            perm = torch.randperm(len(ya), device=PDEV)
            for i in range(0, len(ya), 256):
                b = perm[i:i+256]
                loss = crit(net(Xa[b]), ya[b])
                opt.zero_grad(); loss.backward(); opt.step(); sch.step()
        net.eval()
        with torch.no_grad(): oof[iva] = torch.softmax(net(Xv), 1).cpu().numpy()
    return oof

HEAD = {}
_urut = MASTER.index.tolist()
for t in OKT:
    HEAD[t] = dict(lr=BEST[t][3], knn=INFO[t]["knn"], knn_k=INFO[t]["knn_k"], ncm=INFO[t]["ncm"])
if MLP["enable"]:
    for t in _urut[:MLP["top"]]:
        HEAD[t]["mlp"] = popf1(cv_mlp(feat_of(t, BEST[t][0]), FOLDS[SEED]))
KEPALA = pd.DataFrame(HEAD).T.reindex(_urut)
display(KEPALA.round(4))
KEPALA.to_csv(os.path.join(OUT, "automl2_kepala.csv"))
for h in ("knn", "ncm", "mlp"):
    if h in KEPALA and KEPALA[h].notna().sum() >= 4:
        m = KEPALA[["lr", h]].dropna().astype(float)
        tau = kendalltau(m.lr, m[h]).correlation
        print(f"Kendall tau regresi logistik lawan {h:3s}: {tau:+.3f}  "
              f"(n={len(m)}, juara menurut {h}: {m[h].idxmax()})")
if "mlp" in KEPALA:
    _g = (KEPALA.mlp - KEPALA.lr).dropna()
    print(f"\nMLP lawan regresi logistik: rata-rata {_g.mean():+.4f}, "
          f"menang di {int((_g > NOISE).sum())} dari {len(_g)} kandidat.")
    print("Kalau MLP jarang menang, informasi di fitur beku sudah terpakai habis oleh")
    print("batas linier, dan sisa kenaikan hanya bisa datang dari fine-tune.")

---
# Bagian 7. Visualisasi perbandingan

Delapan gambar, masing-masing menjawab satu pertanyaan. Semuanya disimpan ke
`automl2/gambar/` supaya bisa dipakai di laporan tanpa menjalankan ulang.

In [ ]:
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 9})
FAMS = sorted(MASTER.keluarga.unique())
_cmap = plt.get_cmap("tab20")
WARNA = {f: _cmap(i % 20) for i, f in enumerate(FAMS)}
ANCHOR_T = "siglip2-384" if "siglip2-384" in MASTER.index else JUARA

# Gambar 1. Peringkat 50 kandidat teratas dengan selang kepercayaannya. Tabel
# lengkap semua kandidat ada di automl2_hasil.csv.
# Batang yang selangnya bertumpang tindih dengan juara tidak boleh dianggap
# lebih buruk, seberapa pun jauh urutannya di daftar.
M = MASTER.head(TOP_GAMBAR).sort_values("popf1")
fig, ax = plt.subplots(figsize=(8.5, max(4, 0.22 * len(M))))
y = np.arange(len(M))
_x0 = max(0.0, float(np.percentile(M.ci_lo, 10)) - 0.02)
ax.barh(y, M.popf1 - _x0, left=_x0, color=[WARNA[f] for f in M.keluarga], alpha=0.85)
ax.errorbar(M.popf1, y, xerr=[M.popf1 - M.ci_lo, M.ci_hi - M.popf1], fmt="none",
            ecolor="black", elinewidth=0.8, capsize=2)
ax.scatter(M.popf1_ulang, y, marker="|", s=90, color="black", zorder=3, label="popf1 di fold lain")
ax.axvline(MASTER.loc[JUARA, "ci_lo"], color="crimson", ls="--", lw=0.8,
           label="batas bawah selang juara")
ax.set_yticks(y); ax.set_yticklabels(M.index, fontsize=8)
ax.set_xlim(_x0, min(1.0, M.ci_hi.max() + 0.01))
ax.set_xlabel("popf1 probe tertala (selang bootstrap 95%)")
ax.set_title(f"Gambar 1. {len(M)} backbone teratas dari {len(MASTER)}, warna menurut keluarga")
ax.legend(loc="lower right", fontsize=8)
simpan_gambar(fig, "01_peringkat"); plt.show()

In [ ]:
# Gambar 1b. Keluarga arsitektur. Dengan lebih dari seratus keluarga, pertanyaan
# yang lebih berguna daripada "backbone mana yang terbaik" adalah "keluarga mana
# yang pantas dijelajahi lebih dalam". Titik adalah setiap anggota keluarga,
# batang adalah anggota terbaiknya.
_fb = MASTER.groupby("keluarga").popf1.agg(["max", "median", "count"]).sort_values("max")
_fb = _fb.tail(35)
fig, ax = plt.subplots(figsize=(8.5, max(4, 0.24 * len(_fb))))
ax.barh(range(len(_fb)), _fb["max"], color=[WARNA[f] for f in _fb.index], alpha=0.35)
for i, f in enumerate(_fb.index):
    v = MASTER.popf1[MASTER.keluarga == f]
    ax.scatter(v, np.full(len(v), i), s=12, color=WARNA[f])
ax.set_yticks(range(len(_fb)))
ax.set_yticklabels([f"{f} ({int(n)})" for f, n in zip(_fb.index, _fb["count"])], fontsize=7)
ax.set_xlim(max(0.0, float(_fb["max"].min()) - 0.05), 1.0)
ax.set_xlabel("popf1"); ax.set_title("Gambar 1b. 35 keluarga terbaik (angka = jumlah anggota)")
simpan_gambar(fig, "01b_keluarga"); plt.show()
FAM_TAB = MASTER.groupby("keluarga").agg(terbaik=("popf1", "max"), median=("popf1", "median"),
                                         anggota=("popf1", "size"),
                                         wakil=("popf1", "idxmax")).sort_values("terbaik", ascending=False)
FAM_TAB.to_csv(os.path.join(OUT, "automl2_keluarga.csv"))
display(FAM_TAB.head(25).round(4))

In [ ]:
# Gambar 2. Kenapa penalaan wajib untuk semua kandidat.
# Sumbu x adalah skor dengan setelan tetap (juara v1), sumbu y skor setelah
# ditala. Titik yang jauh di atas diagonal adalah backbone yang diremehkan oleh
# setelan tetap. Di v1 hal inilah yang membuat dinov3-b16 tampak juara sebelum
# ditala.
fig, axs = plt.subplots(1, 2, figsize=(11, 4.4))
ax = axs[0]
ax.scatter(MASTER.popf1_cepat, MASTER.popf1, c=[WARNA[f] for f in MASTER.keluarga], s=28)
lo = np.nanmin([MASTER.popf1_cepat.min(), MASTER.popf1.min()]) - 0.01
ax.plot([lo, 1], [lo, 1], color="gray", lw=0.8)
for t in MASTER.index[:8]:
    ax.annotate(t, (MASTER.popf1_cepat[t], MASTER.popf1[t]), fontsize=7, xytext=(3, 2),
                textcoords="offset points")
ax.set_xlabel("popf1 setelan tetap (C=16, tanpa normalisasi)"); ax.set_ylabel("popf1 tertala")
ax.set_title("Gambar 2a. Setelan tetap lawan tertala")
# Gambar 2b. Setelan mana yang menang: pooling dan normalisasi.
ax = axs[1]
_best_tag = KISI.groupby("tag").popf1.max()
lab, dat = [], []
for (pool, nm), g in KISI.groupby(["pool", "norm"]):
    per = g.groupby("tag").popf1.max()
    lab.append(f"{pool}\n{nm}"); dat.append((per - _best_tag.reindex(per.index)).values)
ax.boxplot(dat, showfliers=False)
ax.set_xticks(range(1, len(lab) + 1)); ax.set_xticklabels(lab)
ax.axhline(0, color="gray", lw=0.8)
ax.set_ylabel("selisih dengan setelan terbaik kandidat itu")
ax.set_title("Gambar 2b. Rugi karena memilih setelan yang salah")
fig.tight_layout(); simpan_gambar(fig, "02_penalaan"); plt.show()

In [ ]:
# Gambar 3. Peta F1 per kelas. Baris diurutkan menurut popf1. Kolom yang pucat
# di hampir semua baris adalah kelas yang sulit bagi SEMUA backbone, dan itu
# petunjuk bahwa masalahnya ada di data, bukan di model.
_PK = PERKELAS.head(TOP_GAMBAR // 5 * 4)
fig, ax = plt.subplots(figsize=(7.5, max(4, 0.22 * len(_PK))))
im = ax.imshow(_PK.values, aspect="auto", cmap="viridis",
               vmin=max(0.6, np.nanmin(_PK.values)), vmax=1.0)
ax.set_xticks(range(NC)); ax.set_xticklabels(CLASSES, rotation=30)
ax.set_yticks(range(len(_PK))); ax.set_yticklabels(_PK.index, fontsize=7)
for i in range(len(_PK)):
    for j in range(NC):
        v = _PK.values[i, j]
        ax.text(j, i, f"{v:.3f}"[1:], ha="center", va="center", fontsize=6,
                color="white" if v < 0.9 else "black")
fig.colorbar(im, ax=ax, fraction=0.03); ax.grid(False)
ax.set_title("Gambar 3. F1 per kelas untuk setiap backbone")
simpan_gambar(fig, "03_perkelas"); plt.show()

In [ ]:
# Gambar 4. Blok lawan strip, dan pertukaran jawi dengan pegon.
# Blok adalah separuh test dan selalu populasi terlemah. Backbone yang kuat di
# blok tetapi biasa saja di strip adalah pasangan ensemble yang menarik untuk
# backbone yang sebaliknya.
fig, axs = plt.subplots(1, 2, figsize=(11, 4.4))
ax = axs[0]
ax.scatter(MASTER.f1_strip, MASTER.f1_blok, c=[WARNA[f] for f in MASTER.keluarga], s=30)
for t in MASTER.index[:10]:
    ax.annotate(t, (MASTER.f1_strip[t], MASTER.f1_blok[t]), fontsize=7, xytext=(3, 2),
                textcoords="offset points")
ax.set_xlabel("macro-F1 populasi strip"); ax.set_ylabel("macro-F1 populasi blok")
ax.set_title("Gambar 4a. Blok lawan strip")
ax = axs[1]
if "jawi_ke_pegon" in MASTER:
    top = MASTER.head(15)
    x = np.arange(len(top))
    ax.bar(x - 0.2, top.jawi_ke_pegon, 0.4, label="jawi ditebak pegon", color="#3b75af")
    ax.bar(x + 0.2, top.pegon_ke_jawi, 0.4, label="pegon ditebak jawi", color="#d9534f")
    ax.set_xticks(x); ax.set_xticklabels(top.index, rotation=60, ha="right", fontsize=7)
    ax.set_ylabel("jumlah baris OOF"); ax.legend(fontsize=8)
ax.set_title("Gambar 4b. Pasangan tersulit, 15 teratas")
fig.tight_layout(); simpan_gambar(fig, "04_blok_strip_jawi_pegon"); plt.show()

In [ ]:
# Gambar 5. Biaya lawan skor. Titik di garis depan Pareto (tidak ada backbone
# lain yang lebih murah DAN lebih baik) adalah satu-satunya pilihan yang masuk
# akal. Kecepatan diukur di GPU sesi ini, jadi angkanya berlaku untuk T4.
fig, axs = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, kol, lab in ((axs[0], "img_per_s", "citra per detik (inferensi, lebih tinggi lebih murah)"),
                     (axs[1], "params_m", "juta parameter")):
    ax.scatter(MASTER[kol], MASTER.popf1, s=12 + 3 * np.sqrt(MASTER.params_m.clip(1)),
               c=[WARNA[f] for f in MASTER.keluarga], alpha=0.85)
    for t in MASTER.index[:10]:
        ax.annotate(t, (MASTER[kol][t], MASTER.popf1[t]), fontsize=7, xytext=(3, 2),
                    textcoords="offset points")
    ax.set_xscale("log"); ax.set_xlabel(lab); ax.set_ylabel("popf1")
# garis Pareto di panel kecepatan
_m = MASTER.sort_values("img_per_s", ascending=False); _best = -1; _pts = []
for t, r in _m.iterrows():
    if r.popf1 > _best: _pts.append((r.img_per_s, r.popf1)); _best = r.popf1
if _pts: axs[0].plot(*zip(*_pts), color="crimson", lw=1, ls="--", label="garis Pareto")
axs[0].legend(fontsize=8)
axs[0].set_title("Gambar 5a. Kecepatan lawan skor"); axs[1].set_title("Gambar 5b. Ukuran lawan skor")
fig.tight_layout(); simpan_gambar(fig, "05_biaya"); plt.show()
PARETO = [t for t, r in _m.iterrows() if (r.img_per_s, r.popf1) in _pts]
print(f"backbone di garis Pareto kecepatan: {PARETO}")

In [ ]:
# Gambar 6. Kurva resolusi per keluarga. Satu backbone yang sama diukur di
# beberapa resolusi. Kurva yang masih naik di ujung kanan berarti resolusi
# yang lebih tinggi masih punya ruang.
_res = [(CAND[t]["name"], t) for t in MASTER.index]
_grp = {}
for nm, t in _res: _grp.setdefault(nm, []).append(t)
fig, ax = plt.subplots(figsize=(7.5, 4.2)); _ada_kurva = False
for nm, ts in _grp.items():
    if len(ts) < 2: continue
    ts = sorted(ts, key=lambda t: CAND[t]["h"])
    ax.plot([CAND[t]["h"] for t in ts], [MASTER.popf1[t] for t in ts], "o-",
            color=WARNA[CAND[ts[0]]["fam"]], label=nm.split(".")[0][:34]); _ada_kurva = True
# SigLIP2 base memakai nama berbeda per resolusi, jadi digabung manual
_sg = [t for t in ("siglip2-224", "siglip2-256", "siglip2-384", "siglip2-512") if t in MASTER.index]
if len(_sg) >= 2:
    ax.plot([CAND[t]["h"] for t in _sg], [MASTER.popf1[t] for t in _sg], "s-",
            color=WARNA["siglip2"], label="siglip2 base"); _ada_kurva = True
_sl = [t for t in ("siglip2L-256", "siglip2L-384") if t in MASTER.index]
if len(_sl) >= 2:
    ax.plot([CAND[t]["h"] for t in _sl], [MASTER.popf1[t] for t in _sl], "s--",
            color=WARNA["siglip2"], label="siglip2 large")
_so = [t for t in ("siglip2-so400m-384", "siglip2-so400m-512") if t in MASTER.index]
if len(_so) >= 2:
    ax.plot([CAND[t]["h"] for t in _so], [MASTER.popf1[t] for t in _so], "s:",
            color=WARNA["siglip2"], label="siglip2 so400m")
ax.set_xlabel("resolusi sisi (piksel)"); ax.set_ylabel("popf1")
ax.set_title("Gambar 6. Kurva resolusi"); ax.legend(fontsize=7)
simpan_gambar(fig, "06_resolusi"); plt.show()

In [ ]:
# Gambar 7. Jurang domain lawan skor, dan perbandingan kepala.
fig, axs = plt.subplots(1, 2, figsize=(11, 4.4))
ax = axs[0]
ax.scatter(MASTER.auc_domain, MASTER.popf1, c=[WARNA[f] for f in MASTER.keluarga], s=30)
for t in MASTER.index[:10]:
    ax.annotate(t, (MASTER.auc_domain[t], MASTER.popf1[t]), fontsize=7, xytext=(3, 2),
                textcoords="offset points")
ax.set_xlabel("AUC train lawan test di ruang fitur (lebih rendah lebih baik)")
ax.set_ylabel("popf1"); ax.set_title("Gambar 7a. Kekuatan lawan kepekaan domain")
ax = axs[1]
_k = KEPALA.head(15)
for i, (h, mk) in enumerate((("lr", "o"), ("knn", "^"), ("ncm", "s"), ("mlp", "D"))):
    if h in _k: ax.scatter(_k[h].astype(float), np.arange(len(_k)), marker=mk, s=22, label=h)
ax.set_yticks(range(len(_k))); ax.set_yticklabels(_k.index, fontsize=7); ax.invert_yaxis()
ax.set_xlabel("popf1"); ax.legend(fontsize=8); ax.set_title("Gambar 7b. Empat jenis kepala")
fig.tight_layout(); simpan_gambar(fig, "07_domain_kepala"); plt.show()

In [ ]:
# Gambar 8. Matriks kebingungan dan kalibrasi juara.
fig, axs = plt.subplots(1, 2, figsize=(11, 4.4))
cm = confusion_matrix(ytrue, OOF[JUARA].argmax(1), labels=range(NC))
cmn = cm / cm.sum(1, keepdims=True)
ax = axs[0]; ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1); ax.grid(False)
for i in range(NC):
    for j in range(NC):
        if cm[i, j]: ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7,
                             color="white" if cmn[i, j] > 0.5 else "black")
ax.set_xticks(range(NC)); ax.set_xticklabels(CLASSES, rotation=30)
ax.set_yticks(range(NC)); ax.set_yticklabels(CLASSES)
ax.set_xlabel("tebakan"); ax.set_ylabel("sebenarnya")
ax.set_title(f"Gambar 8a. Kebingungan {JUARA} (OOF)")
ax = axs[1]
for t in list(MASTER.index[:4]):
    pr = OOF[t]; conf = pr.max(1); acc = (pr.argmax(1) == ytrue)
    edges = np.linspace(0, 1, 11); xs, ys = [], []
    for a, b in zip(edges[:-1], edges[1:]):
        m = (conf > a) & (conf <= b)
        if m.sum() >= 5: xs.append(conf[m].mean()); ys.append(acc[m].mean())
    ax.plot(xs, ys, "o-", ms=3, label=f"{t} (ECE {MASTER.ece[t]:.3f})")
ax.plot([0, 1], [0, 1], color="gray", lw=0.8)
ax.set_xlabel("keyakinan"); ax.set_ylabel("akurasi"); ax.legend(fontsize=7)
ax.set_title("Gambar 8b. Kalibrasi 4 teratas")
fig.tight_layout(); simpan_gambar(fig, "08_kebingungan_kalibrasi"); plt.show()

---
# Bagian 8. Keberagaman dan ensemble

### 8.1 Siapa melengkapi siapa

Ensemble hanya berguna kalau anggotanya salah di tempat yang berbeda.
Proyek ini pernah membuktikannya dengan cara yang mahal: `submission_iw.csv`
merata-ratakan tiga model dari satu backbone yang sama dan turun 0,02054 di
papan. Karena itu, sebelum mencari ensemble, setiap pasangan dari 12
kandidat teratas diukur dengan tiga angka.

Tumpang tindih kesalahan adalah irisan dibagi gabungan himpunan baris yang
salah (Jaccard). Nilai rendah berarti keduanya jarang salah di baris yang
sama. Ketidaksepakatan adalah proporsi baris di mana tebakan keduanya
berbeda. Untung campur adalah popf1 dari rata-rata probabilitas keduanya
dikurangi popf1 anggota terbaiknya. Nilai positif di atas ambang derau
adalah bukti langsung bahwa keduanya saling melengkapi.

In [ ]:
TOPD = MASTER.index[:DIV_TOP].tolist()
n = len(TOPD)
JAC, DIS, GAIN = (np.full((n, n), np.nan) for _ in range(3))
ERR = {t: OOF[t].argmax(1) != ytrue for t in TOPD}
for i, a in enumerate(TOPD):
    for j, b in enumerate(TOPD):
        u = (ERR[a] | ERR[b]).sum()
        if i != j: JAC[i, j] = (ERR[a] & ERR[b]).sum() / max(u, 1)
        DIS[i, j] = (OOF[a].argmax(1) != OOF[b].argmax(1)).mean()
        if i != j:
            GAIN[i, j] = popf1((OOF[a] + OOF[b]) / 2) - max(BEST[a][3], BEST[b][3])
fig, axs = plt.subplots(1, 2, figsize=(12.5, 5.2))
for ax, Mx, ttl, cm_, fmt in ((axs[0], JAC, "tumpang tindih kesalahan (Jaccard)", "magma_r", "{:.2f}"),
                              (axs[1], GAIN * 1000, "untung campur berpasangan (x0,001)", "RdYlGn", "{:+.1f}")):
    vmax = np.nanmax(np.abs(Mx)) if cm_ == "RdYlGn" else None
    im = ax.imshow(Mx, cmap=cm_, vmin=(-vmax if vmax else None), vmax=vmax); ax.grid(False)
    ax.set_xticks(range(n)); ax.set_xticklabels(TOPD, rotation=60, ha="right", fontsize=7)
    ax.set_yticks(range(n)); ax.set_yticklabels(TOPD, fontsize=7)
    for i in range(n):
        for j in range(n):
            if not np.isnan(Mx[i, j]) and i != j:
                ax.text(j, i, fmt.format(Mx[i, j]), ha="center", va="center", fontsize=5.5)
    fig.colorbar(im, ax=ax, fraction=0.04); ax.set_title(ttl)
fig.suptitle("Gambar 9. Keberagaman 12 kandidat teratas")
fig.tight_layout(); simpan_gambar(fig, "09_keberagaman"); plt.show()

_pairs = sorted(((GAIN[i, j], TOPD[i], TOPD[j]) for i in range(n) for j in range(i + 1, n)
                 if not np.isnan(GAIN[i, j])), reverse=True)
PASANGAN = pd.DataFrame([dict(a=a, b=b, untung=g, jaccard=JAC[TOPD.index(a), TOPD.index(b)],
                              keluarga_sama=CAND[a]["fam"] == CAND[b]["fam"])
                         for g, a, b in _pairs])
PASANGAN.to_csv(os.path.join(OUT, "automl2_pasangan.csv"), index=False)
print("10 pasangan dengan untung campur terbesar:")
display(PASANGAN.head(10).round(4))
_sama = PASANGAN.groupby("keluarga_sama").untung.mean()
print("rata-rata untung campur, pasangan keluarga sama lawan berbeda:")
print(_sama.round(4).to_string())

### 8.2 Ensemble probe yang dipilih secara serakah dan digerbang

Mulai dari juara, lalu setiap langkah menambahkan satu kandidat yang paling
menaikkan popf1 rata-rata probabilitas. Kandidat hanya masuk kalau
kenaikannya melewati ambang derau 0,0015. Pencarian ini dilakukan di OOF
seed 42, jadi hasilnya sedikit optimistis. Untuk mengukur seberapa
optimistis, susunan yang terpilih dibekukan lalu diukur ulang di OOF seed 7
dan 2026 yang tidak ikut memilih.

Sebagai pembanding juga diukur fusi awal: fitur anggota-anggota yang sama
digabung (masing-masing dinormalkan L2) lalu dilatih satu regresi logistik.

In [ ]:
pool_e = MASTER.index[:ENS["top"]].tolist()
ENS_M = [JUARA]; cur = BEST[JUARA][3]; LOG = [(JUARA, cur)]
while len(ENS_M) < ENS["max_k"]:
    cand, cs = None, cur
    for t in pool_e:
        if t in ENS_M: continue
        s = popf1(np.mean([OOF[q] for q in ENS_M + [t]], 0))
        if s > cs: cand, cs = t, s
    if cand is None or cs - cur < NOISE: break
    ENS_M.append(cand); cur = cs; LOG.append((cand, cs))
for t, s in LOG: print(f"  + {t:22s} popf1 {s:.4f}")
fam_e = sorted({CAND[t]["fam"] for t in ENS_M})
ENS_S42 = cur
ENS_UL = [popf1(np.mean([OOF_S[q][s] for q in ENS_M], 0)) for s in SEEDS_ULANG]
JUA_UL = [popf1(OOF_S[JUARA][s]) for s in SEEDS_ULANG]
print(f"\nensemble {ENS_M} ({len(fam_e)} keluarga: {fam_e})")
print(f"  seed 42   : ensemble {ENS_S42:.4f} | juara tunggal {BEST[JUARA][3]:.4f}")
for s, a, b in zip(SEEDS_ULANG, ENS_UL, JUA_UL):
    print(f"  seed {s:<5d}: ensemble {a:.4f} | juara tunggal {b:.4f} | untung {a-b:+.4f}")
ENS_OK = len(ENS_M) > 1 and np.mean(ENS_UL) - np.mean(JUA_UL) >= NOISE
print("  -> untung campur " + ("BERTAHAN di fold yang tidak ikut memilih." if ENS_OK else
      "TIDAK bertahan (atau tidak ada anggota kedua). Pakai juara tunggal."))

if len(ENS_M) > 1:
    def _l2np(a): return a / np.maximum(np.linalg.norm(a, axis=1, keepdims=True), 1e-9)
    Xcat = np.hstack([_l2np(feat_of(t, BEST[t][0])) for t in ENS_M])
    r = cv_lr(Xcat, FOLDS[SEED], norms=("raw", "std"), Cs=GRID["C"])
    (nm_f, C_f), best_f = max(((k, popf1(v)) for k, v in r.items()), key=lambda kv: kv[1])
    print(f"\nfusi awal (fitur digabung) dengan anggota yang sama: {best_f:.4f} "
          f"(norm={nm_f}, C={C_f}) lawan fusi akhir {ENS_S42:.4f}")

---
# Bagian 9. Konfirmasi: apakah urutan probe bertahan setelah fine-tune?

Semua angka di atas adalah probe beku. Bagian ini melatih ulang backbone
teratas dari keluarga yang berbeda, dengan anggaran yang sama persis (fold 0,
6 epoch, resep augmentasi v13), lalu membandingkan urutannya dengan urutan
probe. Jangkarnya SigLIP2 @384, yang di fold 0 dengan 18 epoch penuh
mencetak 0,9849 sampai 0,9879; dengan 6 epoch angkanya pasti lebih rendah,
dan itu wajar karena yang dibandingkan adalah urutan, bukan nilai mutlak.

Hanya backbone sampai 130 juta parameter yang ikut, karena backbone yang
lebih besar tidak muat dilatih lima fold dalam satu sesi T4 dan karena itu
tidak bisa menjadi anggota submission.

Empat titik adalah sampel kecil untuk korelasi peringkat. Hasilnya dibaca
sebagai bukti tambahan, bukan sebagai kepastian.

In [ ]:
CW = torch.tensor(cnt.sum() / (NC * np.maximum(cnt, 1)), dtype=torch.float32, device=DEV)
DEGRADE = dict(p_blur=0.70, blur=(0.6, 2.2), p_contrast=0.60, contrast=(0.45, 0.95),
               p_rescale=0.50, rescale=(0.35, 0.80), p_jpeg=0.50, jpeg=(25, 88))
ARJIT = dict(lo=0.65, hi=1.55)
AUG_GEO = T.Compose([
    T.RandomApply([T.RandomRotation(4, fill=255)], p=0.5),
    T.RandomApply([T.RandomAffine(0, translate=(0.02, 0.05), shear=4, fill=255)], p=0.3),
    T.ColorJitter(brightness=0.30, contrast=0.30)])
ERASE = T.RandomErasing(p=0.25, scale=(0.01, 0.06))

def degrade(g):
    d = DEGRADE
    if random.random() < d["p_blur"]:
        g = g.filter(ImageFilter.GaussianBlur(random.uniform(*d["blur"])))
    if random.random() < d["p_contrast"]:
        g = ImageEnhance.Contrast(g).enhance(random.uniform(*d["contrast"]))
    if random.random() < d["p_rescale"]:
        w, h = g.size; f = random.uniform(*d["rescale"])
        g = g.resize((max(8, int(w*f)), max(8, int(h*f))), Image.BILINEAR).resize((w, h), Image.BILINEAR)
    if random.random() < d["p_jpeg"]:
        b = io.BytesIO(); g.convert("L").save(b, "JPEG", quality=random.randint(*d["jpeg"]))
        b.seek(0); g = Image.open(b).convert("L")
    return g

def lb_train(g, H, W):
    w, h = g.size
    f = random.uniform(ARJIT["lo"], ARJIT["hi"])
    w = max(1, int(round(w*f))); g = g.resize((w, h), Image.BILINEAR)
    s = min(H/max(h, 1), W/max(w, 1)) * random.uniform(0.92, 1.08)
    nw, nh = max(1, min(W, int(round(w*s)))), max(1, min(H, int(round(h*s))))
    g = g.resize((nw, nh), Image.BILINEAR)
    c = Image.new("L", (W, H), 255)
    c.paste(g, (random.randint(0, W-nw), random.randint(0, H-nh))); return c

class TrainDS(Dataset):
    def __init__(s, df, r, train):
        s.p = df.path.tolist(); s.y = df.y.tolist(); s.r, s.t = r, train
        s.norm = T.Normalize(r.get("mean", MEAN_DEF), r.get("std", STD_DEF))
    def __len__(s): return len(s.p)
    def __getitem__(s, i):
        g = load_img(s.p[i])
        g = lb_train(degrade(g), s.r["h"], s.r["w"]) if s.t else letterbox(g, s.r["h"], s.r["w"])
        if s.t: g = AUG_GEO(g)
        x = TO_T(g)
        if x.shape[0] == 1: x = x.repeat(3, 1, 1)
        if s.t and random.random() < 0.3: x = (x + torch.randn_like(x) * 0.04).clamp(0, 1)
        x = s.norm(x)
        if s.t: x = ERASE(x)
        return x, s.y[i]

class Net(nn.Module):
    def __init__(s, body, feat, nc):
        super().__init__(); s.body = body; s.head = nn.Linear(feat, nc)
    def forward(s, x): return s.head(s.body(x))

def _depth_of(name, nmax):
    parts = name.split(".")
    for k, tok in enumerate(parts):
        if tok in ("embeddings", "patch_embed", "stem", "cls_token", "pos_embed", "reg_token"): return 0
        if tok in ("layer", "layers", "block", "blocks", "stage", "stages"):
            for q in parts[k+1:k+3]:
                if q.isdigit(): return 1 + int(q)
    return nmax

def param_groups(model, lr, wd, decay):
    """Skala layer-decay dikalikan langsung ke lr tiap grup, karena AdamW tidak
    membaca `lr_scale` dan OneCycleLR menimpa lr tiap grup."""
    ps = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    nmax = 0
    for n, _ in ps:
        parts = n.split(".")
        for k, tok in enumerate(parts):
            if tok in ("layer", "layers", "block", "blocks", "stage", "stages"):
                for q in parts[k+1:k+3]:
                    if q.isdigit(): nmax = max(nmax, 1 + int(q))
    nmax += 1; b = {}
    for n, p in ps:
        b.setdefault((min(_depth_of(n, nmax), nmax), (p.ndim <= 1 or n.endswith(".bias"))), []).append(p)
    return [{"params": pr, "weight_decay": (0.0 if nd else wd), "lr": lr*(decay**(nmax-d))}
            for (d, nd), pr in sorted(b.items())]

def short_ft(c):
    itr, iva = FOLDS[SEED][CONFIRM["fold"]]
    dtr = tr_df.iloc[itr].reset_index(drop=True); dva = tr_df.iloc[iva].reset_index(drop=True)
    got = load_body(c)
    if got is None: return None
    body, dim, mean, std = got
    r = dict(h=c["h"], w=c["w"], mean=mean, std=std)
    seed_all(SEED)
    model = Net(body, dim, NC).to(DEV)
    if hasattr(model.body, "set_grad_checkpointing"):
        try: model.body.set_grad_checkpointing(True)
        except Exception: pass
    micro = 16 if c["h"] <= 384 else 8
    accum = max(1, 32 // micro)
    ld = DataLoader(TrainDS(dtr, r, True), batch_size=micro, shuffle=True,
                    num_workers=CFG["nw"], drop_last=True, pin_memory=(DEV == "cuda"))
    crit = nn.CrossEntropyLoss(weight=CW, label_smoothing=CFG["ls"])
    pg = param_groups(model.body, CONFIRM["lr_body"], CFG["wd"], CONFIRM["layer_decay"])
    pg += [{"params": list(model.head.parameters()), "weight_decay": 0.0, "lr": CONFIRM["lr_head"]}]
    opt = torch.optim.AdamW(pg, lr=CONFIRM["lr_body"], weight_decay=CFG["wd"])
    steps = max(10, CONFIRM["epochs"] * max(1, math.ceil(len(ld) / accum)))
    sch = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[g["lr"] * 2 for g in opt.param_groups], total_steps=steps, pct_start=0.25)
    sc = torch.amp.GradScaler(DEV, enabled=(DEV == "cuda"))
    t0 = time.time()
    try:
        for ep in range(CONFIRM["epochs"]):
            model.train(); opt.zero_grad(set_to_none=True)
            for bi, (x, y) in enumerate(ld):
                x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
                with torch.autocast(DEV, torch.float16, enabled=(DEV == "cuda")):
                    loss = crit(model(x), y) / accum
                sc.scale(loss).backward()
                if (bi + 1) % accum == 0 or (bi + 1) == len(ld):
                    sc.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    sc.step(opt); sc.update(); opt.zero_grad(set_to_none=True)
                    if sch.last_epoch < steps - 1: sch.step()
            print(f"      epoch {ep+1}/{CONFIRM['epochs']} {time.time()-t0:.0f}s", flush=True)
    except Exception as e:
        del model
        if DEV == "cuda": torch.cuda.empty_cache()
        print(f"      !! latih gagal: {type(e).__name__}: {str(e)[:300]}")
        return None
    model.eval(); pr = []
    with torch.no_grad():
        for x, _ in DataLoader(TrainDS(dva, r, False), batch_size=micro * 2, shuffle=False,
                               num_workers=CFG["nw"]):
            with torch.autocast(DEV, torch.float16, enabled=(DEV == "cuda")):
                pr.append(model(x.to(DEV)).float().argmax(1).cpu().numpy())
    del model
    if DEV == "cuda": torch.cuda.empty_cache()
    pred = np.concatenate(pr)
    full = np.full(len(ytrue), -1); full[iva] = pred
    return dict(f1=macro_f1(dva.y.values, pred), popf1=popf1_idx(full, iva),
                jam=(time.time() - t0) / 3600)

FT = {}
_ftc = [os.path.join(OUT, "automl2_finetune.csv")] + (glob.glob("/kaggle/input/**/automl2_finetune.csv",
                                                               recursive=True) if IS_KAGGLE else [])
for _p in _ftc:
    try:
        for t, r in pd.read_csv(_p, index_col=0).iterrows():
            FT.setdefault(t, dict(r))
    except Exception: pass
if FT: print(f"hasil fine-tune dari sesi sebelumnya: {list(FT)}")
if BELUM and not os.environ.get("INFEST_FT_PAKSA"):
    print("Fine-tune pendek DITUNDA ke sesi terakhir, karena masih ada kandidat yang belum")
    print("diekstrak dan pilihan wakil keluarganya bisa berubah. Set INFEST_FT_PAKSA=1 untuk memaksa.")
elif CONFIRM["enable"]:
    # pilih wakil terbaik tiap keluarga yang muat dilatih, jangkar selalu ikut
    pilih, fam_ = [], set()
    if CONFIRM["anchor"] in MASTER.index:
        pilih.append(CONFIRM["anchor"]); fam_.add(CAND[CONFIRM["anchor"]]["fam"])
    for t in MASTER.index:
        if len(pilih) >= CONFIRM["k"]: break
        c = CAND[t]
        if c["fam"] in fam_ or MASTER.params_m[t] > CONFIRM["max_params_m"] or c["src"] != "timm":
            continue
        pilih.append(t); fam_.add(c["fam"])
    print(f"kandidat fine-tune: {pilih} {clk()}")
    for t in pilih:
        c = CAND[t]
        if t in FT: print(f"  {t}: sudah ada dari sesi sebelumnya"); continue
        # perkiraan: 6 epoch ViT-B @384 di T4 sekitar 0,2 jam per fold
        est = 0.20 * (CONFIRM["epochs"] / 6) * (c["h"] * c["w"]) / (384 * 384) * max(c["mult"], 0.5)
        if est * BUDGET["safety"] > rem_h() - BUDGET["reserve_h"]:
            print(f"  {t}: DILEWATI, perkiraan {est:.2f} jam, sisa {rem_h():.2f} jam"); continue
        print(f"  fine-tune pendek {t} @{c['h']} (perkiraan {est:.2f} jam)")
        v = short_ft(c)
        if v is not None:
            FT[t] = v
            print(f"      macro-F1 fold {CONFIRM['fold']} = {v['f1']:.4f} | popf1 {v['popf1']:.4f} "
                  f"| {v['jam']:.2f} jam {clk()}")
    FT = {t: v for t, v in FT.items() if t in BEST}
    if len(FT) >= 2:
        _p = [BEST[t][3] for t in FT]; _f = [FT[t]["f1"] for t in FT]
        rho = spearmanr(_p, _f).correlation if len(FT) >= 3 else np.sign((_p[0]-_p[1])*(_f[0]-_f[1]))
        print(f"\nkorelasi peringkat probe lawan fine-tune: {rho:+.2f} (n={len(FT)})")
        print("  +1 berarti urutannya identik, 0 berarti tidak berhubungan, -1 berarti terbalik.")
        fig, ax = plt.subplots(figsize=(5.2, 4))
        ax.scatter(_p, _f, s=40, c=[WARNA[CAND[t]["fam"]] for t in FT])
        for t in FT: ax.annotate(t, (BEST[t][3], FT[t]["f1"]), fontsize=8, xytext=(3, 2),
                                 textcoords="offset points")
        ax.set_xlabel("popf1 probe tertala"); ax.set_ylabel(f"macro-F1 fine-tune fold {CONFIRM['fold']}")
        ax.set_title("Gambar 10. Probe lawan fine-tune")
        simpan_gambar(fig, "10_probe_vs_finetune"); plt.show()
    pd.DataFrame(FT).T.to_csv(os.path.join(OUT, "automl2_finetune.csv"))
else:
    print("konfirmasi fine-tune dimatikan lewat CONFIRM['enable'].")

---
# Bagian 10. Prediksi test dan `submission.csv`

Setiap kandidat sudah dilatih ulang dengan setelan terbaiknya di seluruh data
latih lalu menebak ke-1.220 citra uji (dikerjakan di Bagian 4.2). Probabilitas semua kandidat disimpan ke
`automl2_test_prob.npz`, sehingga notebook lain (misalnya v15) bisa memakainya
untuk percobaan campur tanpa mengulang ekstraksi.

`submission.csv` ditulis dari ensemble Bagian 8.2 kalau untung campurnya
bertahan di fold yang tidak ikut memilih, dan dari juara tunggal kalau tidak.
Sekali lagi, ini submission probe beku. OOF probe terbaik di proyek ini
sekitar 0,95, sedangkan OOF fine-tune v15 sekitar 0,98, jadi berkas ini
hampir pasti di bawah 0,90090 dan tidak dimaksudkan menggantikan v15.
Kegunaannya dua: sebagai pembanding yang benar-benar berbeda cara
pembuatannya, dan sebagai titik data tentang seberapa jauh probe dari papan.

Kalau ada berkas submission lama yang ikut dilampirkan sebagai input
(misalnya `submission.csv` v15), notebook mencarinya dan mencetak persentase
kesepakatan. Kesepakatan yang rendah dengan v15 berarti berkas ini adalah
kandidat slot kedua yang berguna, karena slot kedua sebaiknya berbeda.

In [ ]:
# prediksi test tiap kandidat sudah dihitung di Bagian 4.2 (TEST_P)
np.savez_compressed(os.path.join(OUT, "automl2_test_prob.npz"),
                    classes=np.array(CLASSES), image_id=te_df.image_id.values,
                    **{t.replace("-", "_"): p.astype(np.float32) for t, p in TEST_P.items()})
np.savez_compressed(os.path.join(OUT, "automl2_oof_prob.npz"), classes=np.array(CLASSES),
                    image_id=tr_df.image_id.values, y=ytrue,
                    **{t.replace("-", "_"): OOF[t].astype(np.float32) for t in OKT})

PAKAI = ENS_M if ENS_OK else [JUARA]
P_TEST = np.mean([TEST_P[t] for t in PAKAI], 0)
sub = pd.DataFrame({"image_id": te_df.image_id, "label": [CLASSES[i] for i in P_TEST.argmax(1)]})
_ref = pd.read_csv(sub_csv)
sub = _ref[["image_id"]].merge(sub, on="image_id", how="left")
assert sub.label.notna().all() or SMOKE, "ada id test tanpa tebakan"
sub.to_csv(os.path.join(WORK, "submission.csv"), index=False)
sub.to_csv(os.path.join(OUT, "submission_automl2.csv"), index=False)
print(f"submission.csv ditulis dari {PAKAI}")
print(f"  sebaran tebakan: {sub.label.value_counts().to_dict()}")

# kesepakatan dengan submission lain yang terlihat dari sini
_lain = []
for root in SEARCH:
    for dp, dn, fs in os.walk(root, followlinks=True):
        dn[:] = [d for d in dn if not d.startswith(("_cache", "_fitur", "automl2", "_imgs"))]
        for f in fs:
            if f.lower().endswith(".csv") and "submission" in f.lower() \
               and f.lower() != "sample_submission.csv":
                p = os.path.join(dp, f)
                if os.path.abspath(p) != os.path.abspath(os.path.join(WORK, "submission.csv")):
                    _lain.append(p)
for p in sorted(set(_lain))[:12]:
    try:
        o = pd.read_csv(p)
        if {"image_id", "label"} <= set(o.columns) and len(o) == len(sub):
            m = sub.merge(o, on="image_id", suffixes=("", "_o"))
            print(f"  sepakat dengan {os.path.relpath(p, '/') if p.startswith('/') else p}: "
                  f"{(m.label == m.label_o).mean()*100:.2f}%")
    except Exception:
        pass

---
# Bagian 11. Ringkasan dan cara membaca

Sel terakhir menulis ringkasan dalam kalimat, dihitung dari angka di atas,
dan menyimpannya ke `automl2_ringkasan.txt`. Semua keputusan di dalamnya
memakai aturan yang ditulis sebelum angkanya ada: selisih di bawah 0,0015
dianggap derau, dua kandidat dianggap setara kalau juara mengalahkan yang
lain di kurang dari 90% sampel bootstrap, dan ensemble hanya dipakai kalau
untungnya bertahan di pembagian fold yang tidak ikut memilihnya.

In [ ]:
L = []
def tulis(s=""): L.append(s); print(s)

tulis("=" * 78)
tulis("RINGKASAN AUTOML VERSI 2")
tulis("=" * 78)
if BELUM:
    tulis(f"!! {len(BELUM)} kandidat BELUM diekstrak. Jalankan sesi lanjutan (Bagian 1.3).")
tulis(f"{len(OKT)} dari {len(CANDIDATES)} kandidat terukur, "
      f"{MASTER.keluarga.nunique()} keluarga. Waktu total {el_h():.2f} jam.")
if SMOKE: tulis("!! MODE SMOKE: bobot acak dan data dipangkas, angka di bawah TIDAK BERARTI.")
tulis("")
j = MASTER.loc[JUARA]
tulis(f"Juara probe: {JUARA} ({CAND[JUARA]['name']} @{CAND[JUARA]['h']}), popf1 {j.popf1:.4f} "
      f"[{j.ci_lo:.4f}, {j.ci_hi:.4f}], di fold lain {j.popf1_ulang:.4f}.")
tulis(f"Setelannya: pooling {BEST[JUARA][0]}, normalisasi {BEST[JUARA][1]}, C={BEST[JUARA][2]:g}.")
tulis(f"Setara dengan juara (tidak bisa dibedakan dengan data ini): "
      f"{[t for t in _setara if t != JUARA]}")
tulis("")
top5 = MASTER.head(5)
tulis("Lima teratas:")
for t, r in top5.iterrows():
    tulis(f"  {t:20s} {r.popf1:.4f}  blok {r.f1_blok:.4f}  strip {r.f1_strip:.4f}  "
          f"AUC domain {r.auc_domain:.3f}  {r.params_m:.0f} jt param  {r.img_per_s:.0f} citra/s")
tulis("")

# pertanyaan-pertanyaan yang diajukan di Bagian 1.2
def _s(t): return MASTER.popf1.get(t, np.nan)
tulis("Jawaban atas pertanyaan di Bagian 1.2:")
if not np.isnan(_s("siglip2-so400m-384")) and not np.isnan(_s("siglip2-384")):
    d = _s("siglip2-so400m-384") - _s("siglip2-384")
    tulis(f"  (a) ukuran SigLIP2: so400m lawan base @384 = {d:+.4f} "
          + ("-> ukuran membantu." if d > NOISE else "-> tidak ada untung yang melewati derau."))
_dv = [t for t in ("dinov3-s16", "dinov3-b16", "dinov3-L16") if t in MASTER.index]
if len(_dv) >= 2:
    tulis("      ukuran DINOv3 @384: " + ", ".join(f"{t} {_s(t):.4f}" for t in _dv))
_dr = [t for t in ("dinov3-b16-256", "dinov3-b16", "dinov3-b16-512") if t in MASTER.index]
if len(_dr) >= 2:
    tulis("  (b) resolusi DINOv3: " + ", ".join(f"{CAND[t]['h']} {_s(t):.4f}" for t in _dr))
    if "dinov3-b16-512" in MASTER.index and "dinov3-b16" in MASTER.index:
        d = _s("dinov3-b16-512") - _s("dinov3-b16")
        tulis(f"      512 lawan 384 = {d:+.4f} "
              + ("-> anggota DINOv3 v15 masih punya ruang di 512." if d > NOISE
                 else "-> 384 sudah cukup untuk DINOv3."))
_pl = KISI.groupby(["tag", "pool"]).popf1.max().unstack()
if "alt" in _pl:
    _g = (_pl["alt"] - _pl["bawaan"]).dropna()
    tulis(f"  (c) pooling alternatif lawan bawaan: menang di {int((_g > NOISE).sum())}, "
          f"kalah di {int((_g < -NOISE).sum())}, seri di {int((_g.abs() <= NOISE).sum())} kandidat.")
    for t in ("siglip2-384", "dinov3-b16", "siglip2-512"):
        if t in _g: tulis(f"      {t}: {_g[t]:+.4f}")
_baru = [c["fam"] for c in CANDIDATES if c["fam"] in ("pe", "aimv2", "clip", "siglip1", "metaformer",
                                                      "maxvit", "efficientnet", "deit", "regnet",
                                                      "efficientvit")]
_bf = MASTER[MASTER.keluarga.isin(set(_baru))]
if len(_bf):
    tb = _bf.popf1.idxmax()
    tulis(f"  (d) keluarga baru terbaik: {tb} {_bf.popf1.max():.4f} "
          f"(selisih dengan juara {_bf.popf1.max() - j.popf1:+.4f})")
if "dinov3-L16-sat" in MASTER.index and "dinov3-L16" in MASTER.index:
    tulis(f"  (e) DINOv3-L satelit lawan web: {_s('dinov3-L16-sat'):.4f} lawan {_s('dinov3-L16'):.4f}")
tulis("")
tulis(f"Bias karena memilih setelan: rata-rata {ULANG.bias_pilih.mean():+.4f}; "
      f"simpangan antar pembagian fold rata-rata {ULANG.sd_fold.mean():.4f}.")
if "mlp" in KEPALA:
    tulis(f"MLP lawan regresi logistik: rata-rata {(KEPALA.mlp - KEPALA.lr).dropna().mean():+.4f}.")
tulis(f"Ensemble probe: {ENS_M}, " + ("DIPAKAI" if ENS_OK else "tidak dipakai")
      + f" (untung di fold lain {np.mean(ENS_UL) - np.mean(JUA_UL):+.4f}).")
if len(PASANGAN):
    p0 = PASANGAN.iloc[0]
    tulis(f"Pasangan paling saling melengkapi: {p0.a} + {p0.b} (untung {p0.untung:+.4f}, "
          f"Jaccard kesalahan {p0.jaccard:.2f}).")
if FT:
    tulis("Fine-tune pendek fold 0: " + ", ".join(f"{t} {v['f1']:.4f}" for t, v in FT.items()))
tulis("")
tulis("CARA MEMBACA:")
tulis("  1. Semua popf1 di sini adalah probe beku. Urutannya yang berguna, nilainya tidak.")
tulis("     SigLIP2 @224: probe 0,9250, fine-tune 0,9735, papan 0,87749.")
tulis("  2. Dua kandidat yang selang bootstrapnya bertumpang tindih belum tentu berbeda.")
tulis("     Lihat kolom p_juara_lebih_baik, bukan urutan barisnya.")
tulis("  3. Backbone baru baru layak dibayar fine-tune penuh (sekitar 3 jam di T4 untuk")
tulis("     5 fold) kalau ia setara juara DAN kesalahannya berbeda (Gambar 9), atau jauh")
tulis("     lebih murah pada skor yang sama (Gambar 5).")
tulis("  4. submission.csv di sini adalah ensemble probe, bukan pengganti v15.")
try:
    with open(os.path.join(OUT, "automl2_ringkasan.txt"), "w") as f: f.write("\n".join(L))
except Exception: pass
print(f"\nberkas keluaran di {OUT}:")
for f in sorted(os.listdir(OUT)):
    if not f.startswith("_"): print("  ", f)
print(f"\n{clk()} selesai.")